# Authentication and Authorization in Production, Explained Simply

> **Who is this for:** developers who are about to put a login page, an API, or an AI feature in front of real users, and want to know how sign-in and permissions work in production without reading every RFC.
>
> **What you will learn:**
> - the difference between authentication ("who are you?") and authorization ("what may you do?"), and the attacks you are defending against
> - passwords done right: slow hashes, NIST rules, rate limits, reset links, magic links and one-time codes
> - MFA and passkeys: a real TOTP implementation checked against the RFC test vectors, and a toy passkey flow
> - sessions vs tokens: cookies, JWTs (built by hand), refresh-token rotation, logout and revocation, the BFF pattern
> - OAuth 2.0 and OpenID Connect: an end-to-end simulation of the authorization code flow with PKCE
> - enterprise SSO (SAML, OIDC, SCIM), API keys, signed requests, webhooks and service-to-service auth
> - authorization models (RBAC, ABAC, ReBAC), policy engines, and tests that catch an IDOR bug
> - the browser side: CORS, CSRF, XSS, CSP and clickjacking, with a CORS decision simulator
> - how to choose an auth provider, the limits you will hit, and a production checklist
>
> **Prerequisites:** basic Python and HTTP. Helpful: [Security for AI applications](08_security_for_ai_applications.ipynb) (section 3 introduces hashes, JWTs, API keys, OAuth and RBAC), [Computer networks for AI apps](07_computer_networks_for_ai_apps.ipynb) (HTTP, TLS) and [Connecting AI to a frontend](10_connecting_ai_to_a_frontend.ipynb) (CORS in a real FastAPI app).
>
> **Time needed:** about 2 hours to read, under a minute to run.
>
> **Runs on:** CPU only, Python standard library only (`hmac`, `hashlib`, `secrets`, `base64`, `json`, `time`, `struct`). No API key, no internet. Every "server" and "identity provider" here is a small Python stand-in.

### How this notebook fits with the rest of the repo

This notebook goes wide on auth. It links to the notebooks below instead of repeating them.

| Notebook | What it already covers | What this notebook adds |
|---|---|---|
| [23 / 08 Security for AI applications](08_security_for_ai_applications.ipynb) | OWASP Top 10, password hashing basics, JWT anatomy, API keys, OAuth idea, RBAC | the full production picture: sessions, refresh rotation, PKCE flow, passkeys, TOTP, ReBAC, providers, limits |
| [12 / 02 Advanced FastAPI](../12_FastAPI/02_advanced_fastapi.ipynb) and [`fastapi_project/auth.py`](../12_FastAPI/fastapi_project/auth.py) | JWT login with the OAuth2 password form in FastAPI | why the password grant is deprecated for third parties, and what to use instead |
| [27 / 02 Securing vibe-coded apps](../27_AI_and_Cybersecurity/02_securing_vibe_coded_applications.ipynb) | plain-text passwords, no login rate limit, IDOR, CORS mistakes, row-level security | the design of each fix in more depth |
| [10 / 16 Agentic workflow patterns](../10_Agents/16_agentic_workflow_patterns_and_deep_research.ipynb) (section 7) | agent identity, scoped short-lived tokens for tools | how those tokens are issued (client credentials, token exchange) |
| [28 / 02 Forward deployed engineering](../28_Applied_AI_and_Forward_Deployed_Engineering/02_forward_deployed_engineering_in_practice.ipynb) (sections 9.1 and 9.4) | webhook verification, SSO with SAML or OIDC, SCIM in customer integrations | multi-tenant SSO routing and JIT provisioning |
| [16 / 06 AI SaaS end to end](../16_Deployment/06_deploying_an_ai_saas_application_end_to_end.ipynb) (sections 2 and 3.3) | tenants, hashed API keys, quotas, billing webhooks | key format, rotation, per-key limits |
| [20 / 07 DevSecOps](../20_Git_and_DevOps/07_devsecops_cicd_supply_chain_and_platform_engineering.ipynb) (section 10) | OIDC federation from CI to cloud | where workload identity fits in service-to-service auth |

**A note on safety.** All demos attack only toy objects that live inside this notebook. Never test these ideas against systems you do not own.

### Table of contents

1. Authentication vs authorization, and the threat model
2. Passwords done right
3. MFA and passkeys
4. Sessions vs tokens
5. OAuth 2.0 and OpenID Connect
6. Enterprise SSO: SAML, OIDC, SCIM
7. API authentication: keys, signed requests, mTLS, workload identity, webhooks
8. Authorization models: RBAC, ABAC, ReBAC and policy engines
9. Browser security around auth: CORS, CSRF, XSS, CSP, clickjacking
10. Choosing an auth provider vs building your own, limits, and the production checklist
11. Service map, how to try it for real, certification topic map
12. Recap, interview questions, further reading

## 0. Setup

Only the standard library. We use a **fake clock** in many demos so that "wait 15 minutes" takes no real time.

In [1]:
import base64, hashlib, hmac, json, secrets, struct, time, random
from urllib.parse import urlencode, urlparse, parse_qs

random.seed(42)  # only used for toy data; all security values come from `secrets`

class FakeClock:
    """A clock we can move forward by hand, so expiry demos run instantly."""
    def __init__(self, start=1_800_000_000):
        self.t = start
    def now(self):
        return self.t
    def advance(self, seconds):
        self.t += seconds

def b64url(data: bytes) -> str:
    """Base64url without '=' padding, as used by JWTs and PKCE."""
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()

def b64url_decode(text: str) -> bytes:
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))

def check(label, condition):
    """Tiny test helper: prints PASS/FAIL and raises if the check fails."""
    print(("PASS " if condition else "FAIL ") + label)
    assert condition, label

print("ready")

ready


## 1. Authentication vs authorization, and the threat model

### 1.1 Two different questions

| | Authentication (authN) | Authorization (authZ) |
|---|---|---|
| Question | Who are you? | What are you allowed to do? |
| Happens | once at login (then remembered by a session or token) | on **every** request |
| Fails with | `401 Unauthorized` (really means "unauthenticated") | `403 Forbidden` (or `404` to hide that the thing exists) |
| Example | password + TOTP code proves you are Alice | Alice may edit doc 7 because she owns it |

**Simple analogy:** at an airport, showing your passport at security is authentication. Your boarding pass deciding which plane and which seat you may use is authorization.

### 1.2 The words you will meet

- **Identity:** the account a person or program has, for example `user_123` or `service:billing`.
- **Credential:** the thing used to prove an identity: a password, a passkey, a TOTP code, an API key, a client certificate.
- **Factor:** a kind of credential. Something you know (password), have (phone, security key), or are (fingerprint, which unlocks a key on your device).
- **Session:** the server remembering "this browser already logged in", usually through a random ID in a cookie.
- **Token:** a string the client sends with each request to prove it is allowed. It is either *opaque* (a random ID the server looks up) or *self-contained* (a signed JWT the server can check without a lookup).
- **Identity provider (IdP):** a service that does login for you and tells your app who the user is (Google, Microsoft Entra ID, Okta, Auth0, Keycloak).

```mermaid
flowchart LR
    U[User] -->|credentials| L[Login: authentication]
    L -->|creates| S[Session or token]
    S -->|sent with every request| R[API request]
    R --> Z{Authorization check:<br/>may this identity do this action<br/>on this resource?}
    Z -->|yes| OK[200 + data]
    Z -->|no| NO[403 / 404]
```

### 1.3 The threat model: what attackers actually do

| Attack | What it is (one sentence) | Main defences (sections) |
|---|---|---|
| Credential stuffing | trying username/password pairs leaked from *other* sites | breached-password checks, rate limits, MFA, passkeys (2, 3) |
| Password spraying | trying a few common passwords against many accounts | blocklist of common passwords, per-IP and global limits (2) |
| Phishing | a fake site tricks the user into typing their password or code | passkeys (bound to the real domain), not SMS (3) |
| Session hijacking | stealing a session cookie or token and replaying it | HttpOnly + Secure cookies, short lifetimes, rotation, binding (4) |
| CSRF | a malicious site makes the user's browser send a request with their cookies | SameSite cookies, CSRF tokens, Fetch Metadata checks (9) |
| XSS token theft | injected JavaScript reads tokens from `localStorage` | HttpOnly cookies, output encoding, CSP (4, 9) |
| IDOR / broken access control | changing `/docs/7` to `/docs/8` returns someone else's data | server-side check on every request, deny by default (8) |
| Token leakage | tokens in logs, URLs, browser history, or chat transcripts | never put tokens in URLs, redact logs, short TTLs (4, 5, 7) |
| Account recovery abuse | taking over an account through "forgot password" or support | single-use expiring reset tokens, notify the user (2) |
| SIM swap | the attacker moves your phone number to their SIM to receive your SMS codes | prefer TOTP or passkeys over SMS (3) |

> **Best practices: section 1**
> - Treat authN and authZ as separate layers. A logged-in user is not automatically allowed anything.
> - Write down your threat model (the table above is a good start) before choosing tools.
> - Return the same error for "no such user" and "wrong password" so attackers cannot list your users.

## 2. Passwords done right

### 2.1 Why password hashes must be slow

**What it is:** you never store the password. You store the output of a *password hashing function*: a one-way function that is deliberately slow and (ideally) uses a lot of memory.

**Simple analogy:** a normal hash like SHA-256 is a revolving door: an attacker who steals your database can push billions of guesses through per second on a GPU. A password hash is a heavy vault door: each guess takes a noticeable fraction of a second and a chunk of memory, so guessing at scale becomes very expensive.

**How it works:**
1. Pick a random **salt** (16 bytes) per user, so two users with the same password get different hashes and precomputed "rainbow tables" are useless.
2. Run the slow function: `hash = KDF(password, salt, cost settings)`.
3. Store `algorithm + settings + salt + hash` together in one string. The settings travel with the hash, so you can raise them later.
4. At login, recompute with the stored salt and settings, and compare with a **constant-time** comparison (`hmac.compare_digest`).

**Which function (OWASP Password Storage Cheat Sheet, as of late 2026):**

| Choice | Minimum settings OWASP recommends | Notes |
|---|---|---|
| **Argon2id** (first choice) | 19 MiB memory, 2 iterations, parallelism 1 | memory-hard; winner of the Password Hashing Competition. Python: `argon2-cffi` |
| **scrypt** (if no Argon2id) | N = 2^17, r = 8, p = 1 | memory-hard; in the stdlib as `hashlib.scrypt` |
| **bcrypt** (legacy systems) | work factor 10 or more | **only the first 72 bytes** of the password are used |
| **PBKDF2-HMAC-SHA256** (when FIPS-140 is required) | 600,000 iterations | not memory-hard; in the stdlib as `hashlib.pbkdf2_hmac` |
| SHA-256, MD5, SHA-1 (plain) | never for passwords | far too fast |

Let us measure the difference. The stdlib has PBKDF2 and scrypt; Argon2id needs a library, so it is shown as a snippet after the demo.

In [2]:
password = b"correct horse battery staple"
salt = secrets.token_bytes(16)

def timed(label, fn, repeats=1):
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    per_call = (time.perf_counter() - start) / repeats
    print(f"{label:<42} {per_call*1000:10.3f} ms per hash  -> about {1/per_call:>12,.0f} guesses/second on this CPU core")
    return per_call

t_sha = timed("SHA-256 (fast, WRONG for passwords)", lambda: hashlib.sha256(salt + password).digest(), repeats=20000)
t_pbk = timed("PBKDF2-HMAC-SHA256, 600,000 iterations", lambda: hashlib.pbkdf2_hmac("sha256", password, salt, 600_000))
# scrypt with OWASP's minimum: N=2**17, r=8 needs 128 * N * r bytes = 128 MiB of memory per guess
t_scr = timed("scrypt N=2^17, r=8, p=1 (128 MiB)", lambda: hashlib.scrypt(password, salt=salt, n=2**17, r=8, p=1, maxmem=256 * 1024 * 1024, dklen=32))

print(f"\nPBKDF2 is ~{t_pbk/t_sha:,.0f}x slower than SHA-256 per guess; scrypt is ~{t_scr/t_sha:,.0f}x slower AND needs 128 MiB per guess.")
print("For one user logging in, a few hundred ms is fine. For an attacker trying billions of guesses, it is ruinous.")

SHA-256 (fast, WRONG for passwords)             0.001 ms per hash  -> about      697,216 guesses/second on this CPU core


PBKDF2-HMAC-SHA256, 600,000 iterations        362.427 ms per hash  -> about            3 guesses/second on this CPU core


scrypt N=2^17, r=8, p=1 (128 MiB)             367.167 ms per hash  -> about            3 guesses/second on this CPU core

PBKDF2 is ~252,690x slower than SHA-256 per guess; scrypt is ~255,995x slower AND needs 128 MiB per guess.
For one user logging in, a few hundred ms is fine. For an attacker trying billions of guesses, it is ruinous.


Now a small but complete "hash and verify" helper that stores the settings with the hash, adds an optional **pepper**, and tells you when a hash needs upgrading.

- **Salt:** random, per user, stored next to the hash. Not secret.
- **Pepper:** one secret for the whole app, stored **outside** the database (a secrets manager or HSM). If only the database leaks, the attacker cannot even start guessing. Here we apply it OWASP's "post-hashing" way: an HMAC over the slow hash. The downside: if you change the pepper, every user must reset their password.

In [3]:
PEPPER = secrets.token_bytes(32)        # in production: load from a secrets manager, never from the DB
SCRYPT_N, SCRYPT_R, SCRYPT_P = 2**14, 8, 1  # demo setting (16 MiB) so the notebook stays fast; use N=2**17 in production

def hash_password(pw: str) -> str:
    salt = secrets.token_bytes(16)
    slow = hashlib.scrypt(pw.encode(), salt=salt, n=SCRYPT_N, r=SCRYPT_R, p=SCRYPT_P, dklen=32)
    peppered = hmac.new(PEPPER, slow, hashlib.sha256).digest()   # post-hash pepper
    return f"scrypt${SCRYPT_N}${SCRYPT_R}${SCRYPT_P}${b64url(salt)}${b64url(peppered)}"

def verify_password(pw: str, stored: str) -> bool:
    algo, n, r, p, salt_b64, hash_b64 = stored.split("$")
    slow = hashlib.scrypt(pw.encode(), salt=b64url_decode(salt_b64), n=int(n), r=int(r), p=int(p), dklen=32)
    candidate = hmac.new(PEPPER, slow, hashlib.sha256).digest()
    return hmac.compare_digest(candidate, b64url_decode(hash_b64))   # constant-time compare

def needs_rehash(stored: str) -> bool:
    """True when the stored settings are weaker than today's. Rehash right after a successful login."""
    _, n, r, p, *_ = stored.split("$")
    return (int(n), int(r), int(p)) < (SCRYPT_N, SCRYPT_R, SCRYPT_P)

h1, h2 = hash_password("hunter2-but-much-longer"), hash_password("hunter2-but-much-longer")
print("stored:", h1)
check("same password, different salts -> different stored strings", h1 != h2)
check("correct password verifies", verify_password("hunter2-but-much-longer", h1))
check("wrong password fails", not verify_password("hunter2-but-much-longer!", h1))
check("an old, weaker hash is flagged for upgrade", needs_rehash("scrypt$1024$8$1$xx$yy"))

stored: scrypt$16384$8$1$yIpNIeiuMXdSIuhWr510Kg$QcJlhK2W0UukvTGcpYMjk3qtcu998N_lWz8woxQMSD8
PASS same password, different salts -> different stored strings
PASS correct password verifies
PASS wrong password fails
PASS an old, weaker hash is flagged for upgrade


In a real Python app you would use a library rather than hand-rolling the format:

```python
# pip install argon2-cffi   (shown, not run)
from argon2 import PasswordHasher
ph = PasswordHasher(memory_cost=19456, time_cost=2, parallelism=1)  # 19 MiB, t=2, p=1 (OWASP minimum)
stored = ph.hash("correct horse battery staple")      # "$argon2id$v=19$m=19456,t=2,p=1$<salt>$<hash>"
ph.verify(stored, "correct horse battery staple")      # raises VerifyMismatchError on a wrong password
if ph.check_needs_rehash(stored):                       # settings went up since this hash was made
    stored = ph.hash("correct horse battery staple")
```

In Node.js the common choices are `argon2` or `@node-rs/argon2`; in Java, Spring Security's `Argon2PasswordEncoder`.

### 2.2 Password rules: NIST SP 800-63B-4

NIST's digital identity guideline (revision 4, final in 2025) flipped many old habits. The main rules for passwords:

| Rule | NIST SP 800-63B-4 says | Why |
|---|---|---|
| Minimum length | **15** characters if the password is the only factor; **8** if it is used with MFA | length beats complexity |
| Maximum length | allow **at least 64** characters; allow spaces and Unicode | let people use passphrases and password managers |
| Composition rules | **SHALL NOT** require "one upper, one digit, one symbol" | they produce `Password1!` |
| Blocklist | **SHALL** check new passwords against common, expected and breached passwords | stops credential stuffing and spraying |
| Periodic rotation | **SHALL NOT** force regular changes; force a change only on evidence of compromise | forced rotation makes passwords weaker |
| Hints and security questions | no hints; no "mother's maiden name" | easy to research |
| Failed attempts | limit consecutive failures on one account to **no more than 100** | stops online guessing (we use a much lower, progressive limit below) |

**Breached-password checks in practice:** services such as Have I Been Pwned's *Pwned Passwords* use **k-anonymity**: you send only the first 5 hex characters of the SHA-1 of the password, get back all suffixes with that prefix, and compare locally. The service never sees the password. Below we simulate it with a tiny offline list.

In [4]:
# A tiny stand-in for a breached-password corpus. Real lists have hundreds of millions of entries.
BREACHED = {hashlib.sha1(p.encode()).hexdigest().upper() for p in
            ["password", "123456789", "qwertyuiop", "iloveyou", "Password1!", "letmein123456789", "correcthorsebatterystaple"]}

def k_anon_range_query(prefix5: str):
    """Stand-in for the remote API: returns every breached hash suffix that starts with prefix5."""
    return [h[5:] for h in BREACHED if h.startswith(prefix5)]

def is_breached(pw: str) -> bool:
    digest = hashlib.sha1(pw.encode()).hexdigest().upper()
    prefix, suffix = digest[:5], digest[5:]
    return suffix in k_anon_range_query(prefix)        # only `prefix` would leave your server

def check_new_password(pw: str, username: str, uses_mfa: bool) -> list:
    """Return a list of problems (empty list = accepted), following NIST SP 800-63B-4."""
    problems = []
    min_len = 8 if uses_mfa else 15
    if len(pw) < min_len:
        problems.append(f"too short: need at least {min_len} characters")
    if len(pw) > 256:
        problems.append("too long for this service (we allow up to 256)")   # generous; NIST says allow >= 64
    if is_breached(pw):
        problems.append("appears in a breach corpus: choose another")
    if username.lower() in pw.lower() or "zerotoai" in pw.lower():
        problems.append("contains your username or the service name")
    return problems   # note: NO 'must contain a symbol' rule

for pw, mfa in [("Password1!", True), ("correcthorsebatterystaple", False), ("alice-rocks-2026-x", False),
                ("short", True), ("blue kettle under the stairs", False)]:
    print(f"{pw!r:34} mfa={mfa!s:5} -> {check_new_password(pw, 'alice', mfa) or 'accepted'}")

'Password1!'                       mfa=True  -> ['appears in a breach corpus: choose another']
'correcthorsebatterystaple'        mfa=False -> ['appears in a breach corpus: choose another']
'alice-rocks-2026-x'               mfa=False -> ['contains your username or the service name']
'short'                            mfa=True  -> ['too short: need at least 8 characters']
'blue kettle under the stairs'     mfa=False -> accepted


### 2.3 Rate limiting and lockout

Online guessing is stopped by limiting attempts. Two layers work together:

1. **Per account:** after a few failures, add a growing delay (1 s, 2 s, 4 s ...). OWASP advises tying the counter to the *account*, not the IP, so an attacker with many IPs is still slowed. Avoid permanent hard lockouts: they let an attacker lock out real users on purpose (a denial of service).
2. **Per IP / per device and global:** credential stuffing tries *one* password on *many* accounts, so per-account counters never trip. A per-IP limit and a global "failed logins per minute" alarm catch it. Real systems add bot detection, CAPTCHAs after suspicious signals, and checks for impossible travel.

In [5]:
class LoginLimiter:
    """Progressive delay per account + a fixed window per IP. Uses the fake clock."""
    def __init__(self, clock, free_failures=3, max_delay=900, ip_limit=20, ip_window=60):
        self.clock, self.free, self.max_delay = clock, free_failures, max_delay
        self.ip_limit, self.ip_window = ip_limit, ip_window
        self.fails = {}        # account -> (count, locked_until)
        self.ip_hits = {}      # ip -> list of timestamps

    def allowed(self, account, ip):
        now = self.clock.now()
        hits = [t for t in self.ip_hits.get(ip, []) if now - t < self.ip_window]
        self.ip_hits[ip] = hits
        if len(hits) >= self.ip_limit:
            return False, "ip rate limited"
        count, locked_until = self.fails.get(account, (0, 0))
        if now < locked_until:
            return False, f"account delayed for {locked_until - now}s"
        return True, "ok"

    def record(self, account, ip, success):
        now = self.clock.now()
        self.ip_hits.setdefault(ip, []).append(now)
        if success:
            self.fails.pop(account, None)            # reset on success
            return
        count = self.fails.get(account, (0, 0))[0] + 1
        delay = 0 if count <= self.free else min(2 ** (count - self.free - 1), self.max_delay)
        self.fails[account] = (count, now + delay)

clock = FakeClock()
lim = LoginLimiter(clock)

print("Brute force on one account: the attacker tries once per second for 60 seconds")
timeline = ""
for second in range(60):
    ok, why = lim.allowed("alice", "10.0.0.5")
    if ok:
        lim.record("alice", "10.0.0.5", success=False)
    timeline += "T" if ok else "."
    clock.advance(1)
print("  " + timeline)
print(f"  T = guess tried, . = blocked. {timeline.count('T')} guesses in 60 s instead of 60; the gaps double each time.")
clock.advance(3600)   # let the per-IP window and delays expire before the next demo

print("\nCredential stuffing: one IP, one guess per account, 30 accounts:")
blocked = 0
for i in range(30):
    ok, why = lim.allowed(f"user{i}", "203.0.113.9")
    if ok:
        lim.record(f"user{i}", "203.0.113.9", success=False)
    else:
        blocked += 1
print(f"  {30 - blocked} guesses got through, {blocked} were blocked by the per-IP limit")

Brute force on one account: the attacker tries once per second for 60 seconds
  TTTTT.T...T.......T...............T.........................
  T = guess tried, . = blocked. 9 guesses in 60 s instead of 60; the gaps double each time.

Credential stuffing: one IP, one guess per account, 30 accounts:
  20 guesses got through, 10 were blocked by the per-IP limit


### 2.4 Password reset with single-use, expiring tokens

**How it works:**

```mermaid
sequenceDiagram
    participant U as User
    participant A as App
    participant E as Email
    U->>A: POST /forgot {email}
    A->>A: token = random 32 bytes, store SHA-256(token), expiry 15 min, unused
    A-->>U: "If that email exists, we sent a link" (same answer either way)
    A->>E: link https://app/reset?token=...
    U->>A: POST /reset {token, new_password}
    A->>A: look up SHA-256(token): exists? not expired? not used?
    A->>A: set new hash, mark token used, revoke all sessions, notify user
```

Key details:
- Store only a **hash** of the token, so a database leak does not hand out reset links. A fast SHA-256 is fine here because the token is 256 random bits, not a guessable password.
- Short expiry (15 to 60 minutes), **single use**, and invalidate older reset tokens when a new one is issued.
- After a reset, **log out every session** and email the user "your password was changed".
- Same response whether the email exists or not (no user enumeration).
- Put the token in the POST body of the reset form, and set `Referrer-Policy: no-referrer` on the reset page so the token in the URL does not leak to third-party scripts or links.

In [6]:
class ResetService:
    def __init__(self, clock, ttl=15 * 60):
        self.clock, self.ttl = clock, ttl
        self.users = {"alice@example.com": {"pw": hash_password("old password that is long"), "sessions": {"s1", "s2"}}}
        self.tokens = {}   # sha256(token) -> {"email", "exp", "used"}
        self.outbox = []   # stand-in for an email provider

    def request_reset(self, email):
        if email in self.users:
            for rec in self.tokens.values():          # invalidate older outstanding tokens
                if rec["email"] == email:
                    rec["used"] = True
            token = secrets.token_urlsafe(32)
            self.tokens[hashlib.sha256(token.encode()).hexdigest()] = {"email": email, "exp": self.clock.now() + self.ttl, "used": False}
            self.outbox.append((email, f"https://app.example/reset?token={token}"))
        return "If that email is registered, a reset link is on its way."   # identical either way

    def reset(self, token, new_password):
        rec = self.tokens.get(hashlib.sha256(token.encode()).hexdigest())
        if rec is None or rec["used"] or self.clock.now() > rec["exp"]:
            return "invalid or expired link"
        rec["used"] = True
        user = self.users[rec["email"]]
        user["pw"] = hash_password(new_password)
        user["sessions"].clear()                      # log out everywhere
        self.outbox.append((rec["email"], "Your password was changed. Not you? Contact support."))
        return "password updated"

clock = FakeClock()
svc = ResetService(clock)
print(svc.request_reset("alice@example.com"))
print(svc.request_reset("nobody@example.com"), " <- same message, no enumeration")
token = parse_qs(urlparse(svc.outbox[0][1]).query)["token"][0]

check("valid token works once", svc.reset(token, "a brand new long passphrase") == "password updated")
check("all sessions were revoked", svc.users["alice@example.com"]["sessions"] == set())
check("the same token cannot be reused", svc.reset(token, "attacker password here!!") == "invalid or expired link")

svc.request_reset("alice@example.com")
token2 = parse_qs(urlparse(svc.outbox[-1][1]).query)["token"][0]
clock.advance(16 * 60)
check("an expired token is rejected", svc.reset(token2, "too late passphrase!!") == "invalid or expired link")

If that email is registered, a reset link is on its way.
If that email is registered, a reset link is on its way.  <- same message, no enumeration


PASS valid token works once
PASS all sessions were revoked
PASS the same token cannot be reused
PASS an expired token is rejected


### 2.5 Email verification, magic links and one-time codes

All three reuse the reset-token pattern above (random, hashed, expiring, single use):

| Method | How it works | Good for | Watch out for |
|---|---|---|---|
| **Email verification** | send a link or code at sign-up; mark the email verified when used | proving the user owns the address before trusting it (for example before linking accounts by email) | do not auto-link a social login to an existing account by an *unverified* email |
| **Magic link** | email a one-click sign-in link | low-friction consumer apps | email scanners that "click" links can consume them: make the link open a page with a "Sign in" button that POSTs; tie it to the browser that asked when you can |
| **Email or SMS OTP code** | email/text a 6-digit code | mobile apps, people who use a different device for email | short codes are guessable: rate limit attempts (NIST requires this), expire in minutes, and issuing a new code must not reset the failure counter |

**Limits to plan for:** email and SMS cost money and have sending limits. For example, Supabase Auth's built-in email sender allows only 2 emails per hour until you configure your own SMTP provider. Use a transactional email service (SES, SendGrid, Postmark, Resend and similar) with SPF, DKIM and DMARC set up, or your links land in spam.

> **Best practices: section 2**
> - Argon2id (or scrypt), salted, settings stored with the hash; rehash on login when settings go up.
> - Minimum 15 characters (8 with MFA), no composition rules, no forced rotation, blocklist breached passwords.
> - Progressive delays per account plus per-IP and global limits; avoid hard lockouts that attackers can trigger.
> - Reset and magic-link tokens: random, hashed at rest, short-lived, single use; revoke sessions after a reset.
> - Watch the 72-byte input limit if you are stuck with bcrypt.

## 3. MFA and passkeys

**What MFA is:** multi-factor authentication asks for two different kinds of proof, for example a password (something you know) and a code from your phone (something you have). A stolen password alone is then not enough.

| Second factor | Phishing resistant? | Notes |
|---|---|---|
| SMS code | no | SIM swap, number porting, interception; NIST SP 800-63B-4 treats phone (PSTN) delivery as a *restricted* option |
| Email code | no | as strong as the email account |
| TOTP app (Google Authenticator, Authy, 1Password ...) | no (a fake site can relay the code within 30 s) | cheap, offline, standard (RFC 6238) |
| Push approval | no, and prone to "MFA fatigue" spam | use number matching |
| **Passkey / security key (WebAuthn)** | **yes** | the browser binds the login to the real domain |

### 3.1 TOTP from scratch (RFC 6238)

**What it is:** a time-based one-time password. Your phone and the server share a secret key. Both compute a 6-digit code from the key and the current 30-second time window, so the codes match without any network call.

**How it works:**
1. `counter = floor(unix_time / 30)`
2. `mac = HMAC-SHA1(key, counter as 8-byte big-endian)` (this is HOTP, RFC 4226)
3. *Dynamic truncation:* take the low 4 bits of the last byte as an offset, read 4 bytes there, drop the top bit.
4. `code = that number mod 10^digits`

We implement it and check it against **all 18 test vectors in RFC 6238 Appendix B** (SHA-1, SHA-256 and SHA-512, 8 digits).

In [7]:
def hotp(key: bytes, counter: int, digits: int = 6, digest=hashlib.sha1) -> str:
    mac = hmac.new(key, struct.pack(">Q", counter), digest).digest()   # 8-byte big-endian counter
    offset = mac[-1] & 0x0F                                            # dynamic truncation
    number = struct.unpack(">I", mac[offset:offset + 4])[0] & 0x7FFFFFFF
    return str(number % 10 ** digits).zfill(digits)

def totp(key: bytes, unix_time: int, step: int = 30, digits: int = 6, digest=hashlib.sha1) -> str:
    return hotp(key, unix_time // step, digits, digest)

# RFC 6238 Appendix B: the seeds are the ASCII digits "1234567890" repeated to 20, 32 and 64 bytes
seeds = {"SHA1": (b"12345678901234567890", hashlib.sha1),
         "SHA256": (b"12345678901234567890" + b"123456789012", hashlib.sha256),
         "SHA512": (b"1234567890" * 6 + b"1234", hashlib.sha512)}
vectors = [(59, "94287082", "46119246", "90693936"), (1111111109, "07081804", "68084774", "25091201"),
           (1111111111, "14050471", "67062674", "99943326"), (1234567890, "89005924", "91819424", "93441116"),
           (2000000000, "69279037", "90698825", "38618901"), (20000000000, "65353130", "77737706", "47863826")]

ok = 0
for t, *expected in vectors:
    for (name, (seed, dg)), exp in zip(seeds.items(), expected):
        got = totp(seed, t, digits=8, digest=dg)
        ok += got == exp
        print(f"T={t:<12} {name:<7} expected {exp} got {got} {'ok' if got == exp else 'MISMATCH'}")
check(f"all {len(vectors) * 3} RFC 6238 test vectors match", ok == len(vectors) * 3)

T=59           SHA1    expected 94287082 got 94287082 ok
T=59           SHA256  expected 46119246 got 46119246 ok
T=59           SHA512  expected 90693936 got 90693936 ok
T=1111111109   SHA1    expected 07081804 got 07081804 ok
T=1111111109   SHA256  expected 68084774 got 68084774 ok
T=1111111109   SHA512  expected 25091201 got 25091201 ok
T=1111111111   SHA1    expected 14050471 got 14050471 ok
T=1111111111   SHA256  expected 67062674 got 67062674 ok
T=1111111111   SHA512  expected 99943326 got 99943326 ok
T=1234567890   SHA1    expected 89005924 got 89005924 ok
T=1234567890   SHA256  expected 91819424 got 91819424 ok
T=1234567890   SHA512  expected 93441116 got 93441116 ok
T=2000000000   SHA1    expected 69279037 got 69279037 ok
T=2000000000   SHA256  expected 90698825 got 90698825 ok
T=2000000000   SHA512  expected 38618901 got 38618901 ok
T=20000000000  SHA1    expected 65353130 got 65353130 ok
T=20000000000  SHA256  expected 77737706 got 77737706 ok
T=20000000000  SHA512  expected

Verifying a code in production needs three extra rules: accept a **small window** (one step either side) for clock drift, **remember the last step used** so the same code cannot be replayed, and **rate limit** attempts (a 6-digit code has only a million values). Enrolment uses an `otpauth://` URI that the user scans as a QR code.

In [8]:
class TOTPVerifier:
    def __init__(self, clock, key):
        self.clock, self.key, self.last_step = clock, key, -1

    def verify(self, code: str) -> bool:
        current = self.clock.now() // 30
        for step in (current - 1, current, current + 1):            # +-30 s drift window
            if step > self.last_step and hmac.compare_digest(hotp(self.key, step), code):
                self.last_step = step                                # block replays of this or older steps
                return True
        return False

key = secrets.token_bytes(20)
secret_b32 = base64.b32encode(key).decode().rstrip("=")
print("Enrolment URI (shown as a QR code):")
print(f"  otpauth://totp/ZeroToAI:alice@example.com?secret={secret_b32}&issuer=ZeroToAI&algorithm=SHA1&digits=6&period=30\n")

clock = FakeClock()
v = TOTPVerifier(clock, key)
phone_code = totp(key, clock.now())
check("code from the phone verifies", v.verify(phone_code))
check("the same code cannot be replayed", not v.verify(phone_code))
clock.advance(60)
check("a code from a phone 25 s behind still verifies (drift window)", v.verify(totp(key, clock.now() - 25)))
clock.advance(300)
check("an old code is rejected", not v.verify(phone_code))

Enrolment URI (shown as a QR code):
  otpauth://totp/ZeroToAI:alice@example.com?secret=WCCYT6TZZR4BH4UTADM44TROSK2EQPB5&issuer=ZeroToAI&algorithm=SHA1&digits=6&period=30

PASS code from the phone verifies
PASS the same code cannot be replayed
PASS a code from a phone 25 s behind still verifies (drift window)
PASS an old code is rejected


### 3.2 Passkeys and WebAuthn, explained simply

**What it is:** a passkey is a key pair created by your device (phone, laptop, password manager or security key) for **one website**. The private key never leaves the device (or your synced, end-to-end encrypted password manager). The website stores only the **public key**. WebAuthn is the W3C browser API for this (Level 3 became a W3C Recommendation in August 2026); FIDO2/CTAP is how the browser talks to authenticators.

**Simple analogy:** the website gives you a padlock challenge; only your key can produce the right answer, and your key refuses to answer for any shop that is not the one it was cut for.

**How it works (login):**

```mermaid
sequenceDiagram
    participant S as Server (relying party)
    participant B as Browser
    participant D as Device (authenticator)
    S->>B: random challenge + rpId "example.com"
    B->>D: sign(challenge, origin, rpId)
    Note over D: user unlocks with fingerprint/face/PIN<br/>finds the key for rpId, none for look-alike domains
    D-->>B: signature over (authenticatorData with rpId hash + sign counter, hash of clientData with challenge + origin)
    B-->>S: credential id + signature
    S->>S: verify with stored PUBLIC key, check challenge, origin, rpId, counter
```

**Why it beats phishing:** the browser itself puts the real origin into the signed data, and the authenticator only has keys for the real `rpId`. A phishing page at `examp1e.com` gets nothing usable.

The toy below uses an **HMAC stand-in for the signature. This is NOT how WebAuthn works**: with HMAC the server would have to know the device's secret. Real WebAuthn uses public-key signatures (usually ES256 or EdDSA), so the server holds only a public key that is useless to a thief. The toy only shows the *flow*: challenge, origin binding, replay protection and the sign counter.

In [9]:
class ToyAuthenticator:
    """Stands in for a phone/security key. STAND-IN: uses HMAC instead of a real public-key signature."""
    def __init__(self):
        self.keys, self.counter = {}, 0          # rpId -> secret
    def register(self, rp_id):
        self.keys[rp_id] = secrets.token_bytes(32)
        return {"credential_id": b64url(secrets.token_bytes(16)), "verify_key": self.keys[rp_id]}  # real life: a PUBLIC key
    def sign(self, rp_id, client_data: bytes):
        if rp_id not in self.keys:
            return None                          # no passkey for look-alike domains
        self.counter += 1
        auth_data = hashlib.sha256(rp_id.encode()).digest() + struct.pack(">I", self.counter)
        sig = hmac.new(self.keys[rp_id], auth_data + hashlib.sha256(client_data).digest(), hashlib.sha256).digest()
        return auth_data, sig

def browser_get_assertion(device, page_origin, challenge):
    """The BROWSER writes the true origin of the page into clientData; the page cannot lie about it."""
    rp_id = urlparse(page_origin).hostname
    client_data = json.dumps({"type": "webauthn.get", "challenge": challenge, "origin": page_origin}).encode()
    result = device.sign(rp_id, client_data)
    return None if result is None else (client_data, *result)

class RelyingParty:
    def __init__(self, rp_id, origin):
        self.rp_id, self.origin, self.creds, self.pending = rp_id, origin, {}, set()
    def start_login(self):
        c = b64url(secrets.token_bytes(32)); self.pending.add(c); return c
    def finish_login(self, cred, assertion):
        if assertion is None: return "no assertion"
        client_data, auth_data, sig = assertion
        cd = json.loads(client_data)
        if cd["challenge"] not in self.pending: return "unknown or reused challenge"
        self.pending.discard(cd["challenge"])                       # single use
        if cd["origin"] != self.origin: return "wrong origin"
        if auth_data[:32] != hashlib.sha256(self.rp_id.encode()).digest(): return "wrong rpId"
        counter = struct.unpack(">I", auth_data[32:36])[0]
        expected = hmac.new(cred["verify_key"], auth_data + hashlib.sha256(client_data).digest(), hashlib.sha256).digest()
        if not hmac.compare_digest(expected, sig): return "bad signature"
        if counter <= cred.get("last_counter", 0): return "counter went backwards: possible cloned key"
        cred["last_counter"] = counter
        return "logged in"

device = ToyAuthenticator()
rp = RelyingParty("example.com", "https://example.com")
cred = device.register("example.com")

ch = rp.start_login()
good = browser_get_assertion(device, "https://example.com", ch)
check("real site: login works", rp.finish_login(cred, good) == "logged in")
check("replaying the same assertion fails", rp.finish_login(cred, good) == "unknown or reused challenge")
ch = rp.start_login()
check("phishing site examp1e.com gets no assertion at all", browser_get_assertion(device, "https://examp1e.com", ch) is None)

PASS real site: login works
PASS replaying the same assertion fails
PASS phishing site examp1e.com gets no assertion at all


**What you can use:** do not implement WebAuthn verification by hand. Use a maintained library (`py_webauthn` for Python, `@simplewebauthn/server` for Node, `webauthn4j` for Java) or an auth provider that supports passkeys (most in section 10 do).

**Practical points:**
- Offer passkeys *alongside* passwords at first; let users add a passkey after login ("create a passkey for faster sign-in").
- Use **conditional UI** (passkey autofill in the username field) so users do not need to know the word "passkey".
- **Synced passkeys** (Apple, Google, Microsoft, password managers) survive a lost phone. **Device-bound** keys (hardware security keys) are stronger for admins.
- Let users register more than one passkey, and show them a list they can revoke.

### 3.3 Recovery codes

When a user loses their second factor, recovery codes let them back in. Generate around 10 random codes, show them **once**, store them **hashed**, and make each one single use.

In [10]:
def make_recovery_codes(n=10):
    alphabet = "ABCDEFGHJKMNPQRSTVWXYZ23456789"            # no 0/O/1/I/L confusion
    codes = ["-".join("".join(secrets.choice(alphabet) for _ in range(5)) for _ in range(2)) for _ in range(n)]
    salt = secrets.token_bytes(16)
    # hash slowly (demo uses 20k PBKDF2 iterations to stay fast; reuse your password-hash settings in production)
    stored = {hashlib.pbkdf2_hmac("sha256", c.encode(), salt, 20_000).hex() for c in codes}
    return codes, salt, stored

def use_recovery_code(code, salt, stored):
    h = hashlib.pbkdf2_hmac("sha256", code.strip().upper().encode(), salt, 20_000).hex()
    if h in stored:
        stored.discard(h)            # single use
        return True
    return False

codes, salt, stored = make_recovery_codes()
print("Show these once, then never again:", codes[:3], "...")
check("a code works once", use_recovery_code(codes[0], salt, stored))
check("the same code fails the second time", not use_recovery_code(codes[0], salt, stored))
print(f"{len(stored)} unused codes left; warn the user when few remain")

Show these once, then never again: ['CA7PQ-ZGGA3', '6STSD-JNPKE', 'T2482-RK2ZB'] ...


PASS a code works once
PASS the same code fails the second time
9 unused codes left; warn the user when few remain


> **Best practices: section 3**
> - Offer passkeys first, TOTP second, SMS only as a fallback (and never as the only recovery path for admins).
> - TOTP: drift window of +-1 step, no replays, rate limit attempts, store the secret encrypted.
> - Use a WebAuthn library or provider; check challenge, origin, rpId and counter on the server.
> - Recovery codes: random, shown once, hashed, single use; notify the user every time one is used.
> - Require **step-up** (re-authentication) before sensitive actions: changing email, password, MFA, payout details.

## 4. Sessions vs tokens

After login, the app must remember the user. There are two families:

| | Server-side session | Self-contained token (JWT) |
|---|---|---|
| What the client holds | a random session ID (opaque) in a cookie | a signed JSON document |
| Where the truth lives | session store (database, Redis) | inside the token, checked by signature |
| Logout / revoke | delete the row: instant | hard: the token stays valid until it expires (needs a denylist or short TTL) |
| Each request costs | one store lookup (fast with Redis) | a signature check, no lookup |
| Size | ~32-64 bytes | often 500 bytes to several KB |
| Best for | web apps with one backend | APIs called by many services, mobile apps, short-lived access tokens from an IdP |

**Simple analogy:** a session is a cloakroom ticket: just a number, and the cloakroom keeps your coat. A JWT is a signed letter of introduction: anyone who trusts the signature can read it without phoning you, but you cannot take it back once it is out.

Many production systems mix them: a session cookie for the browser, and short-lived JWT access tokens between services.

### 4.1 Server-side sessions and cookie attributes

| Attribute | What it does | Recommended |
|---|---|---|
| `HttpOnly` | JavaScript cannot read the cookie, so XSS cannot steal it | always for session cookies |
| `Secure` | sent only over HTTPS | always |
| `SameSite` | `Strict`: never sent on cross-site requests; `Lax`: sent on top-level GET navigations only; `None`: always (requires `Secure`) | `Lax` (or `Strict` for high-value apps). Some browsers default to `Lax` when it is missing |
| `Domain` | which hosts get the cookie; leaving it out means "this exact host only" | leave it out unless subdomains truly need it |
| `Path` | URL path prefix | `/` |
| `Max-Age` / `Expires` | lifetime; without it the cookie is deleted when the browser closes | match your session policy |
| `__Host-` name prefix | the browser enforces `Secure`, `Path=/` and no `Domain` | recommended by OWASP for session cookies |

**Session rules (OWASP Session Management Cheat Sheet):** session IDs with at least 64 bits of entropy from a secure random generator; a **new session ID on login** and on privilege change (prevents *session fixation*: an attacker plants a known ID before you log in); an **idle timeout** (roughly 2-5 minutes for high-value apps, 15-30 minutes for low-risk ones) and an **absolute timeout** (for example 4-8 hours for a workday app); delete the session on the server at logout.

**Limits:** browsers only promise about **4096 bytes per cookie** (name, value and attributes together) and around 50 cookies per domain (RFC 6265 minimums). Keep cookies tiny; put data in the session store, not in the cookie.

In [11]:
def set_cookie(name, value, *, http_only=True, secure=True, same_site="Lax", path="/", domain=None, max_age=None):
    parts = [f"{name}={value}", f"Path={path}"]
    if domain: parts.append(f"Domain={domain}")
    if max_age is not None: parts.append(f"Max-Age={max_age}")
    if secure: parts.append("Secure")
    if http_only: parts.append("HttpOnly")
    if same_site: parts.append(f"SameSite={same_site}")
    return "; ".join(parts)

def lint_cookie(header: str) -> list:
    """Return warnings for a session cookie's Set-Cookie header."""
    name = header.split("=", 1)[0]
    attrs = {a.split("=")[0].strip().lower(): (a.split("=", 1)[1] if "=" in a else True) for a in header.split(";")[1:]}
    w = []
    if "httponly" not in attrs: w.append("missing HttpOnly (XSS can read it)")
    if "secure" not in attrs: w.append("missing Secure (sent over plain HTTP)")
    if str(attrs.get("samesite", "")).lower() not in ("lax", "strict"): w.append("SameSite should be Lax or Strict")
    if name.startswith("__Host-") and ("domain" in attrs or attrs.get("path") != "/" or "secure" not in attrs):
        w.append("__Host- cookies need Secure, Path=/ and no Domain")
    if len(header.encode()) > 4096: w.append(f"{len(header.encode())} bytes: over the 4096-byte cookie limit")
    return w or ["looks good"]

good = set_cookie("__Host-session", secrets.token_urlsafe(32), max_age=8 * 3600)
bad = set_cookie("session", "x" * 5000, http_only=False, secure=False, same_site=None, domain="example.com")
print(good, "\n ->", lint_cookie(good))
print(bad[:60] + "...", "\n ->", lint_cookie(bad))

__Host-session=EMvMm52aUE_BGFqkB5CqcjaIHZUWcWFsPOQ6LOwSbrE; Path=/; Max-Age=28800; Secure; HttpOnly; SameSite=Lax 
 -> ['looks good']
session=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx... 
 -> ['missing HttpOnly (XSS can read it)', 'missing Secure (sent over plain HTTP)', 'SameSite should be Lax or Strict', '5036 bytes: over the 4096-byte cookie limit']


In [12]:
class SessionStore:
    """A server-side session store with rotation, idle and absolute timeouts (a dict stands in for Redis)."""
    def __init__(self, clock, idle=30 * 60, absolute=8 * 3600):
        self.clock, self.idle, self.absolute, self.rows = clock, idle, absolute, {}

    def create(self, user=None):
        sid = secrets.token_urlsafe(32)                      # 256 bits of randomness
        now = self.clock.now()
        self.rows[sid] = {"user": user, "created": now, "seen": now}
        return sid

    def login(self, old_sid, user):
        self.rows.pop(old_sid, None)                         # ROTATE: never keep a pre-login ID
        return self.create(user)

    def get(self, sid):
        row = self.rows.get(sid)
        if not row: return None
        now = self.clock.now()
        if now - row["seen"] > self.idle or now - row["created"] > self.absolute:
            self.rows.pop(sid); return None
        row["seen"] = now
        return row["user"]

    def logout(self, sid):
        self.rows.pop(sid, None)

clock = FakeClock()
store = SessionStore(clock)
planted = store.create()                      # attacker gets a pre-login session ID and plants it in the victim's browser
real = store.login(planted, "alice")          # victim logs in
check("session fixation blocked: the planted ID is not logged in", store.get(planted) is None)
check("the new session works", store.get(real) == "alice")
clock.advance(31 * 60)
check("idle timeout ends the session", store.get(real) is None)
s2 = store.login(store.create(), "alice"); store.logout(s2)
check("logout deletes the session on the server", store.get(s2) is None)

PASS session fixation blocked: the planted ID is not logged in
PASS the new session works
PASS idle timeout ends the session
PASS logout deletes the session on the server


The same idea in FastAPI (shown, not run):

```python
from fastapi import FastAPI, Response, Request, HTTPException
app = FastAPI()

@app.post("/login")
def login(form: LoginForm, request: Request, response: Response):
    user = authenticate(form.email, form.password)           # slow hash verify + rate limit
    old = request.cookies.get("__Host-session")
    sid = sessions.login(old, user.id)                        # rotate the ID on login
    response.set_cookie("__Host-session", sid, httponly=True, secure=True,
                        samesite="lax", path="/", max_age=8 * 3600)
    return {"ok": True}

def current_user(request: Request):
    user = sessions.get(request.cookies.get("__Host-session"))
    if user is None:
        raise HTTPException(401)
    return user
```

### 4.2 JWTs: build one by hand

**What it is:** a JSON Web Token (RFC 7519) is three base64url parts joined by dots: `header.payload.signature`. The header names the algorithm, the payload holds **claims**, and the signature proves nobody changed the first two parts.

| Claim | Meaning | Must check? |
|---|---|---|
| `iss` | issuer: who made the token | yes, exact match |
| `aud` | audience: which API it is for | yes, must include *your* API |
| `sub` | subject: the user or client ID | used as the identity |
| `exp` / `nbf` / `iat` | expires / not before / issued at (Unix seconds) | yes, with a small clock-skew leeway |
| `jti` | unique token ID | needed for denylists and replay checks |
| `scope` / `roles` | what the token allows | used by authorization |

`HS256` uses one shared secret (HMAC-SHA256) to sign and verify. `RS256`, `ES256` and `EdDSA` sign with a private key and verify with a public key, which is what identity providers use so that anyone can verify without being able to mint tokens. We build HS256 with `hmac`. The payload is **only encoded, not encrypted**: anyone can read it, so never put secrets in it.

In [13]:
def jwt_sign(claims: dict, key: bytes, kid: str = None) -> str:
    header = {"alg": "HS256", "typ": "JWT"} | ({"kid": kid} if kid else {})
    signing_input = b64url(json.dumps(header, separators=(",", ":")).encode()) + "." + \
                    b64url(json.dumps(claims, separators=(",", ":")).encode())
    sig = hmac.new(key, signing_input.encode(), hashlib.sha256).digest()
    return signing_input + "." + b64url(sig)

class JWTError(Exception):
    pass

def jwt_verify(token: str, key: bytes, *, issuer: str, audience: str, now: int, leeway: int = 30, denylist=frozenset()) -> dict:
    try:
        h64, p64, s64 = token.split(".")
        header = json.loads(b64url_decode(h64)); claims = json.loads(b64url_decode(p64))
    except Exception:
        raise JWTError("malformed token")
    if header.get("alg") != "HS256":                     # ALLOW-LIST the algorithm; never trust the header's choice
        raise JWTError(f"algorithm {header.get('alg')!r} not allowed")
    expected = hmac.new(key, f"{h64}.{p64}".encode(), hashlib.sha256).digest()
    if not hmac.compare_digest(expected, b64url_decode(s64)):
        raise JWTError("bad signature")
    if claims.get("iss") != issuer: raise JWTError("wrong issuer")
    aud = claims.get("aud"); aud = aud if isinstance(aud, list) else [aud]
    if audience not in aud: raise JWTError("wrong audience")
    if "exp" not in claims or now > claims["exp"] + leeway: raise JWTError("expired or no exp")
    if now + leeway < claims.get("nbf", 0): raise JWTError("not valid yet")
    if claims.get("jti") in denylist: raise JWTError("revoked")
    return claims

KEY = secrets.token_bytes(32)          # 256-bit random key: as long as the hash output
clock = FakeClock()
claims = {"iss": "https://auth.example", "aud": "notes-api", "sub": "user_123", "scope": "notes:read",
          "iat": clock.now(), "exp": clock.now() + 600, "jti": secrets.token_hex(8)}
token = jwt_sign(claims, KEY)
print("JWT:", token[:70], "...")
print("header :", b64url_decode(token.split(".")[0]).decode())
print("payload:", b64url_decode(token.split(".")[1]).decode(), " <- readable by anyone")
print(f"size   : {len(token)} bytes")
check("valid token verifies", jwt_verify(token, KEY, issuer="https://auth.example", audience="notes-api", now=clock.now())["sub"] == "user_123")

JWT: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL2F1dGguZXhhb ...
header : {"alg":"HS256","typ":"JWT"}
payload: {"iss":"https://auth.example","aud":"notes-api","sub":"user_123","scope":"notes:read","iat":1800000000,"exp":1800000600,"jti":"3f42aa46e709f0b2"}  <- readable by anyone
size   : 275 bytes
PASS valid token verifies


### 4.3 The classic JWT pitfalls, demonstrated

The OWASP JWT Cheat Sheet and RFC 8725 (JWT Best Current Practices) list the same mistakes again and again. Each check below should raise an error, and our verifier does.

In [14]:
def expect_reject(label, fn):
    try:
        fn(); print("FAIL (accepted!) " + label); raise AssertionError(label)
    except JWTError as e:
        print(f"PASS rejected: {label:<48} -> {e}")

V = dict(issuer="https://auth.example", audience="notes-api", now=clock.now())

# 1. alg=none: an attacker removes the signature and says "no algorithm"
h = b64url(json.dumps({"alg": "none", "typ": "JWT"}).encode())
p = b64url(json.dumps(claims | {"sub": "admin", "scope": "notes:admin"}).encode())
expect_reject("alg=none with no signature", lambda: jwt_verify(f"{h}.{p}.", KEY, **V))

# 2. tampered payload, original signature
parts = token.split(".")
expect_reject("payload edited to sub=admin", lambda: jwt_verify(f"{parts[0]}.{p}.{parts[2]}", KEY, **V))

# 3. token for another API (audience confusion)
other = jwt_sign(claims | {"aud": "billing-api"}, KEY)
expect_reject("token minted for billing-api", lambda: jwt_verify(other, KEY, **V))

# 4. token from another issuer that happens to share the key
expect_reject("wrong issuer", lambda: jwt_verify(jwt_sign(claims | {"iss": "https://evil.example"}, KEY), KEY, **V))

# 5. expired token, and a token with no exp at all
expect_reject("expired 1 hour ago", lambda: jwt_verify(token, KEY, **(V | {"now": clock.now() + 3600})))
no_exp = {k: v for k, v in claims.items() if k != "exp"}
expect_reject("no exp claim (would live forever)", lambda: jwt_verify(jwt_sign(no_exp, KEY), KEY, **V))

# 6. revoked via a jti denylist (for example after logout or a password reset)
expect_reject("jti on the denylist", lambda: jwt_verify(token, KEY, **(V | {"denylist": {claims["jti"]}})))

# Why the alg allow-list matters: a NAIVE verifier that trusts the header
def naive_verify(tok, key):
    h64, p64, s64 = tok.split(".")
    if json.loads(b64url_decode(h64))["alg"] == "none":
        return json.loads(b64url_decode(p64))          # BUG: trusts the attacker's header
    raise ValueError("not shown")
print("\nNaive verifier accepts the forged token and returns:", naive_verify(f"{h}.{p}.", KEY)["sub"], "<- account takeover")

PASS rejected: alg=none with no signature                       -> algorithm 'none' not allowed
PASS rejected: payload edited to sub=admin                      -> bad signature
PASS rejected: token minted for billing-api                     -> wrong audience
PASS rejected: wrong issuer                                     -> wrong issuer
PASS rejected: expired 1 hour ago                               -> expired or no exp
PASS rejected: no exp claim (would live forever)                -> expired or no exp
PASS rejected: jti on the denylist                              -> revoked

Naive verifier accepts the forged token and returns: admin <- account takeover


**Weak secrets.** An HS256 token is an offline puzzle: anyone holding one token can guess the key on their own computer as fast as their hardware allows, with no rate limit. A key like `secret123` falls instantly to a word list.

In [15]:
weak_key = b"secret123"
leaked = jwt_sign(claims, weak_key)
h64, p64, s64 = leaked.split(".")
wordlist = [b"password", b"changeme", b"jwt-secret", b"secret", b"secret1", b"secret123", b"supersecret"]  # real lists: millions
for guess in wordlist:
    if hmac.compare_digest(hmac.new(guess, f"{h64}.{p64}".encode(), hashlib.sha256).digest(), b64url_decode(s64)):
        print(f"cracked the HS256 key offline: {guess.decode()!r} -> the attacker can now mint admin tokens")
        break
print("Fix: a 32-byte random key from secrets.token_bytes(32) or a secrets manager, or switch to RS256/ES256/EdDSA.")

cracked the HS256 key offline: 'secret123' -> the attacker can now mint admin tokens
Fix: a 32-byte random key from secrets.token_bytes(32) or a secrets manager, or switch to RS256/ES256/EdDSA.


### Access tokens, bearer tokens and refresh tokens in one picture

**What a bearer token is:** a token where *whoever holds it* can use it, with no extra proof of who they are. The name comes from "pay the bearer": like cash or a concert ticket, the holder gets in. The client sends it in the HTTP header `Authorization: Bearer <token>` (defined in RFC 6750). Most access tokens today are bearer tokens, so they must be kept secret, sent only over HTTPS, and never put in URLs or logs.

**Access token:** the short-lived key for calling APIs (typically 5 to 60 minutes). It can be **opaque** (a random string the API checks with the auth server or a database, via token introspection) or **self-contained** (a JWT the API checks by signature, issuer, audience and expiry). Short lifetimes limit the damage if one leaks.

**Refresh token:** a long-lived token used only at the auth server's token endpoint to get a new access token, so the user does not have to log in again. It is never sent to your APIs. Because it is powerful, it is stored more carefully (an HttpOnly cookie, the OS keychain on mobile, or server-side in a BFF), **rotated** on every use, and the whole token family is revoked if an old one is reused (next section).

**How they work together:**

```mermaid
sequenceDiagram
    participant C as Client app
    participant A as Auth server
    participant R as API (resource server)
    C->>A: log in (for example authorization code + PKCE)
    A-->>C: access token (10 min) + refresh token (days)
    C->>R: GET /notes with Authorization Bearer access token
    R-->>C: 200 OK
    Note over C,R: 10 minutes later the access token expires
    C->>R: GET /notes with the expired token
    R-->>C: 401 with WWW-Authenticate Bearer error invalid_token
    C->>A: POST /token grant_type refresh_token
    A-->>C: new access token + new refresh token (old one now dead)
    C->>R: retry GET /notes with the new token
    R-->>C: 200 OK
```

| | Access token | Refresh token |
|---|---|---|
| Sent to | your APIs, in `Authorization: Bearer` | only the auth server's `/token` endpoint |
| Lifetime | minutes | days to weeks (sliding or absolute) |
| Format | JWT or opaque | usually opaque |
| If stolen | usable until it expires | usable to mint new access tokens until revoked, hence rotation and reuse detection |
| Store in a browser | memory, or let a BFF keep it server-side | HttpOnly Secure cookie, or keep it server-side (BFF) |

**Bearer vs sender-constrained tokens:** a bearer token works for anyone who steals it. **Sender-constrained** tokens are bound to a key the client holds (DPoP, RFC 9449, or mTLS-bound tokens, RFC 8705), so a stolen token alone is useless. These are used for high-value APIs such as banking.

**The demo below** shows an API that checks bearer tokens and returns proper 401 responses, and a client that refreshes automatically on a 401 and retries once.

**Best practices:** keep access tokens short-lived; send them only in the `Authorization` header over HTTPS; never log them or put them in URLs; on the API, check signature, `exp`, `iss` and `aud` on every request and answer 401 with `WWW-Authenticate: Bearer error="invalid_token"`; on the client, refresh once on that 401 and send the user to log in if refreshing fails; rotate refresh tokens and detect reuse.

In [16]:
import hmac, hashlib, json, base64, secrets, time

SECRET = secrets.token_bytes(32)          # demo signing key for access tokens (HS256-style)
ACCESS_TTL = 2                            # seconds, tiny on purpose so we can watch it expire

def _b64(b): return base64.urlsafe_b64encode(b).rstrip(b"=").decode()
def make_access(sub, now):
    body = _b64(json.dumps({"sub": sub, "exp": now + ACCESS_TTL}).encode())
    sig = _b64(hmac.new(SECRET, body.encode(), hashlib.sha256).digest())
    return f"{body}.{sig}"

class DemoAuthServer:
    """Issues access tokens and rotating refresh tokens."""
    def __init__(self): self.refresh = {}            # refresh token -> (user, still_valid)
    def login(self, user, now):
        rt = secrets.token_urlsafe(24); self.refresh[rt] = (user, True)
        return {"access_token": make_access(user, now), "token_type": "Bearer",
                "expires_in": ACCESS_TTL, "refresh_token": rt}
    def token(self, refresh_token, now):
        user, valid = self.refresh.get(refresh_token, (None, False))
        if not valid: return {"error": "invalid_grant"}
        self.refresh[refresh_token] = (user, False)    # rotate: the old one is now dead
        new_rt = secrets.token_urlsafe(24); self.refresh[new_rt] = (user, True)
        return {"access_token": make_access(user, now), "token_type": "Bearer",
                "expires_in": ACCESS_TTL, "refresh_token": new_rt}

def api_get_notes(headers, now):
    """A resource server: checks the bearer token on every request."""
    auth = headers.get("Authorization", "")
    if not auth.startswith("Bearer "):
        return 401, {"WWW-Authenticate": 'Bearer realm="api"'}, None
    token = auth[len("Bearer "):]
    try:
        body, sig = token.split(".")
        good = _b64(hmac.new(SECRET, body.encode(), hashlib.sha256).digest())
        claims = json.loads(base64.urlsafe_b64decode(body + "=" * (-len(body) % 4)))
    except Exception:
        return 401, {"WWW-Authenticate": 'Bearer error="invalid_token"'}, None
    if not hmac.compare_digest(sig, good):
        return 401, {"WWW-Authenticate": 'Bearer error="invalid_token", error_description="bad signature"'}, None
    if claims["exp"] <= now:
        return 401, {"WWW-Authenticate": 'Bearer error="invalid_token", error_description="expired"'}, None
    return 200, {}, (["note A", "note B"] if claims["sub"] == "ada" else [])

class DemoClient:
    """Keeps tokens; on a 401 with invalid_token it refreshes once and retries."""
    def __init__(self, auth): self.auth, self.tokens, self.log = auth, None, []
    def login(self, user, now): self.tokens = self.auth.login(user, now)
    def get_notes(self, now):
        status, hdrs, data = api_get_notes({"Authorization": f"Bearer {self.tokens['access_token']}"}, now)
        self.log.append((round(now - t0), "GET /notes", status, hdrs.get("WWW-Authenticate", "")))
        if status == 401 and "invalid_token" in hdrs.get("WWW-Authenticate", ""):
            new = self.auth.token(self.tokens["refresh_token"], now)
            self.log.append((round(now - t0), "POST /token (refresh)", "invalid_grant" if "error" in new else 200, ""))
            if "error" in new: return status, None              # refresh failed: send the user to log in
            self.tokens = new
            status, hdrs, data = api_get_notes({"Authorization": f"Bearer {self.tokens['access_token']}"}, now)
            self.log.append((round(now - t0), "GET /notes (retry)", status, ""))
        return status, data

auth_server = DemoAuthServer(); client = DemoClient(auth_server)
t0 = time.time()
client.login("ada", t0)
print("1) fresh access token:   ", client.get_notes(t0))
print("2) after it expired:     ", client.get_notes(t0 + 5))   # expired -> refresh -> retry succeeds
print("3) no Authorization:     ", api_get_notes({}, t0)[:2])
print("4) reused refresh token: ", auth_server.token("not-a-valid-refresh-token", t0))
print("\nrequest log (seconds since login, call, status, WWW-Authenticate):")
for row in client.log: print("   ", row)

1) fresh access token:    (200, ['note A', 'note B'])
2) after it expired:      (200, ['note A', 'note B'])
3) no Authorization:      (401, {'WWW-Authenticate': 'Bearer realm="api"'})
4) reused refresh token:  {'error': 'invalid_grant'}

request log (seconds since login, call, status, WWW-Authenticate):
    (0, 'GET /notes', 200, '')
    (5, 'GET /notes', 401, 'Bearer error="invalid_token", error_description="expired"')
    (5, 'POST /token (refresh)', 200, '')
    (5, 'GET /notes (retry)', 200, '')


**Where to keep tokens in a browser, and long lifetimes:**

| Storage | XSS can steal it? | CSRF risk? | Verdict |
|---|---|---|---|
| `localStorage` / `sessionStorage` | **yes**: any injected script can read it | no (not sent automatically) | avoid for long-lived tokens |
| JS memory (a variable) | yes, while the page is open, but it is gone on reload | no | OK for short access tokens in an SPA |
| `HttpOnly; Secure; SameSite` cookie | no | yes, so pair with SameSite + CSRF defences (section 9) | **preferred for browsers** |
| Mobile app | use the OS secure storage: iOS Keychain, Android Keystore-backed storage | n/a | never plain shared preferences or files |

Keep **access tokens short** (5 to 15 minutes is common) and use refresh tokens to get new ones. A one-day JWT that leaks is a one-day account takeover.

**Size limits:** JWTs grow with every claim. Big tokens (many roles or groups) can exceed the 4096-byte cookie limit or a proxy's header size limit (often 8-16 KB for all headers combined; check your load balancer). Keep claims minimal: put a user ID and a few scopes in the token, and look up the rest server-side.

### 4.4 Access tokens + refresh tokens, with rotation and reuse detection

**How it works:** the client gets a short-lived **access token** (used on every API call) and a long-lived **refresh token** (used only at the token endpoint to get new tokens). With **rotation**, every refresh returns a *new* refresh token and the old one stops working. With **reuse detection**, if an *old* refresh token ever shows up again, someone copied it, so the server revokes the whole token **family** and forces a fresh login. RFC 9700 asks that refresh tokens for public clients (SPAs, mobile apps) are either rotated like this or bound to the client (sender-constrained, for example with DPoP).

In [17]:
class RefreshTokenService:
    def __init__(self, clock, ttl=30 * 24 * 3600):
        self.clock, self.ttl = clock, ttl
        self.tokens = {}            # sha256(rt) -> {"family", "user", "used", "exp"}
        self.revoked_families = set()

    def _issue(self, user, family):
        rt = secrets.token_urlsafe(32)
        self.tokens[hashlib.sha256(rt.encode()).hexdigest()] = {"family": family, "user": user, "used": False, "exp": self.clock.now() + self.ttl}
        access = jwt_sign({"iss": "https://auth.example", "aud": "notes-api", "sub": user,
                           "exp": self.clock.now() + 600, "jti": secrets.token_hex(8)}, KEY)
        return access, rt

    def login(self, user):
        return self._issue(user, family=secrets.token_hex(8))

    def refresh(self, rt):
        rec = self.tokens.get(hashlib.sha256(rt.encode()).hexdigest())
        if rec is None or rec["family"] in self.revoked_families or self.clock.now() > rec["exp"]:
            return "rejected: log in again"
        if rec["used"]:                                  # an OLD token came back: theft
            self.revoked_families.add(rec["family"])
            return "REUSE DETECTED: whole family revoked"
        rec["used"] = True
        return self._issue(rec["user"], rec["family"])

clock = FakeClock()
rts = RefreshTokenService(clock)
access1, rt1 = rts.login("alice")
stolen = rt1                                         # malware copies rt1
access2, rt2 = rts.refresh(rt1)                      # the real app refreshes first -> gets rt2
print("attacker uses the stolen rt1 :", rts.refresh(stolen))
print("real app uses rt2            :", rts.refresh(rt2))
check("after reuse, nobody can refresh until a fresh login", rts.refresh(rt2) == "rejected: log in again")

attacker uses the stolen rt1 : REUSE DETECTED: whole family revoked
real app uses rt2            : rejected: log in again
PASS after reuse, nobody can refresh until a fresh login


### 4.5 Logout and revocation

| Technique | How | Trade-off |
|---|---|---|
| Delete the server session | remove the row in the session store | instant; needs a store lookup per request |
| Short access-token TTL | 5-15 min JWTs; revoke the *refresh* token at logout | a stolen access token works until it expires |
| Denylist by `jti` | store revoked `jti`s until their `exp` (Redis with TTL is ideal) | adds a lookup, but only of a small set |
| Token version per user | store `token_version` on the user; put it in the token; bump it on "log out everywhere" or password reset | one cheap lookup; kills all of a user's tokens |
| Introspection (RFC 7662) | the API asks the auth server "is this token active?" | always current; adds latency, so cache briefly |
| OIDC logout | RP-initiated logout, back-channel logout to tell apps a session ended | needed when one IdP session serves many apps |

### 4.6 The BFF pattern for single-page apps

**What it is:** a *backend for frontend* is a small server that sits next to your SPA. It does the OAuth flow, keeps the tokens **on the server**, and gives the browser only an `HttpOnly` session cookie. The browser never sees an access or refresh token, so XSS cannot steal one. The IETF draft "OAuth 2.0 for Browser-Based Applications" recommends this as the most secure option for SPAs.

```mermaid
flowchart LR
    B[Browser SPA] -- "HttpOnly session cookie only" --> BFF[BFF server<br/>same site as the SPA]
    BFF -- "OAuth code + PKCE,<br/>tokens kept here" --> AS[Authorization server]
    BFF -- "Bearer access token" --> API[APIs]
```

When to use it: any SPA handling sensitive data. When not: a public, read-only API, or a native mobile app (which uses the system browser + PKCE and stores tokens in the Keychain/Keystore).

> **Best practices: section 4**
> - Browser apps: a server-side session or a BFF, with a `__Host-` `HttpOnly; Secure; SameSite=Lax` cookie. Rotate the ID at login.
> - JWTs: allow-list algorithms; always check signature, `iss`, `aud`, `exp`; short lifetimes; small payloads; no secrets in claims.
> - Use a vetted library (`PyJWT` or `joserfc`/Authlib for Python, `jose` for Node) rather than hand-rolled code like ours.
> - Refresh tokens: rotate, detect reuse, revoke the family; bind to the client where possible.
> - Plan logout: decide how fast a revoked user must lose access, then pick TTLs and a revocation method to match.

## 5. OAuth 2.0 and OpenID Connect

**What they are:** OAuth 2.0 (RFC 6749) is a standard for **delegated authorization**: letting an app act on a user's behalf at another service *without* getting the user's password (for example "let this app read my Google Calendar"). OpenID Connect (OIDC) is a thin layer on top that adds **authentication**: an **ID token** that tells the app who the user is. "Sign in with Google" is OIDC.

**Simple analogy:** OAuth is a hotel key card. Reception (the authorization server) checks your ID once and gives the cleaner (the app) a card that opens only room 214, only today. The cleaner never sees your passport, and the card can be cancelled.

### 5.1 The four roles

| Role | Who | Example |
|---|---|---|
| Resource owner | the user | Alice |
| Client | the app that wants access | your note-taking app |
| Authorization server (AS) | issues tokens after the user logs in and consents | Google, Entra ID, Okta, Auth0, Keycloak |
| Resource server (RS) | the API holding the data; accepts access tokens | Google Calendar API, your own API |

### 5.2 Authorization code flow with PKCE

This is the flow to use for nearly every app with a user: web apps, SPAs (ideally via a BFF), mobile and desktop apps.

```mermaid
sequenceDiagram
    participant U as User's browser
    participant C as Client app
    participant AS as Authorization server
    participant RS as API (resource server)
    C->>C: code_verifier = random, code_challenge = BASE64URL(SHA256(verifier)), state, nonce = random
    C->>U: redirect to AS /authorize?response_type=code&client_id&redirect_uri&scope=openid notes:read&state&nonce&code_challenge&code_challenge_method=S256
    U->>AS: user logs in (password, passkey, MFA) and consents
    AS->>U: redirect to redirect_uri?code=...&state=...
    U->>C: code + state
    C->>C: check state matches (stops login CSRF)
    C->>AS: POST /token code + code_verifier (+ client secret if confidential)
    AS->>AS: SHA256(code_verifier) == stored challenge? code unused, unexpired? redirect_uri exact match?
    AS-->>C: access_token, id_token, refresh_token
    C->>C: verify id_token: signature, iss, aud, exp, nonce
    C->>RS: GET /notes Authorization: Bearer access_token
```

Why each random value exists:
- **`state`**: ties the response to the request in *this* browser, so an attacker cannot inject their own code (CSRF on the callback).
- **PKCE `code_verifier` / `code_challenge`** (RFC 7636): even if someone intercepts the code (a malicious app on the phone, logs, a referrer header), they cannot redeem it without the verifier that never left the client. RFC 9700 makes PKCE mandatory for public clients and recommended for all.
- **`nonce`**: binds the ID token to this login, stopping ID-token replay.

First, PKCE S256 checked against the example in RFC 7636 Appendix B:

In [18]:
def pkce_challenge(verifier: str) -> str:
    return b64url(hashlib.sha256(verifier.encode("ascii")).digest())

rfc_verifier = "dBjftJeZ4CVP-mB92K27uhbUJU1p1r_wW1gFWFOEjXk"
check("RFC 7636 Appendix B example", pkce_challenge(rfc_verifier) == "E9Melhoa2OwvFrEMTJguCHaoeK1t8URWbuGJSstw-cM")

new_verifier = secrets.token_urlsafe(48)          # 64 chars; RFC 7636 allows 43-128
print("verifier :", new_verifier, len(new_verifier))
print("challenge:", pkce_challenge(new_verifier))

PASS RFC 7636 Appendix B example
verifier : tdsubt2ByATufs7E6zJ1D9_otgwyQjNF63xuo6G_PadBB-GqZ6u8QoEZ5iSuir5i 64
challenge: wiFJYVRKzN5tdeklth7KX17jwxstQ7Dyg8uiC7awBG8


Now an end-to-end simulation with a **mock authorization server**. It is a plain Python class, not a real server, and it signs ID tokens with our HS256 helper for simplicity (real providers use RS256/ES256 and publish public keys, see 5.5).

In [19]:
class MockAuthServer:
    ISSUER = "https://auth.example"
    def __init__(self, clock):
        self.clock, self.key = clock, secrets.token_bytes(32)
        self.clients = {"notes-spa": {"redirect_uris": ["https://notes.example/callback"], "public": True}}
        self.codes = {}

    def authorize(self, params, logged_in_user):
        c = self.clients.get(params.get("client_id"))
        if c is None or params.get("redirect_uri") not in c["redirect_uris"]:     # EXACT match, never prefix
            return {"error": "invalid client or redirect_uri (shown to the user, never redirected)"}
        if params.get("code_challenge_method") != "S256" or not params.get("code_challenge"):
            return {"error": "PKCE S256 required"}
        code = secrets.token_urlsafe(32)
        self.codes[code] = {"client_id": params["client_id"], "redirect_uri": params["redirect_uri"], "user": logged_in_user,
                            "challenge": params["code_challenge"], "nonce": params.get("nonce"),
                            "scope": params.get("scope", ""), "exp": self.clock.now() + 60, "used": False}
        return {"redirect": params["redirect_uri"] + "?" + urlencode({"code": code, "state": params["state"]})}

    def token(self, form):
        rec = self.codes.get(form.get("code"))
        if rec is None or rec["used"] or self.clock.now() > rec["exp"]:
            return {"error": "invalid_grant"}
        rec["used"] = True                                                       # codes are single use
        if form.get("client_id") != rec["client_id"] or form.get("redirect_uri") != rec["redirect_uri"]:
            return {"error": "invalid_grant"}
        if pkce_challenge(form.get("code_verifier", "")) != rec["challenge"]:
            return {"error": "invalid_grant (PKCE verifier does not match)"}
        now = self.clock.now()
        id_token = jwt_sign({"iss": self.ISSUER, "aud": rec["client_id"], "sub": rec["user"], "nonce": rec["nonce"],
                             "iat": now, "exp": now + 300, "email": f"{rec['user']}@example.com"}, self.key)
        access = jwt_sign({"iss": self.ISSUER, "aud": "notes-api", "sub": rec["user"], "scope": rec["scope"],
                           "iat": now, "exp": now + 600, "jti": secrets.token_hex(8)}, self.key)
        return {"access_token": access, "id_token": id_token, "token_type": "Bearer", "expires_in": 600,
                "refresh_token": secrets.token_urlsafe(32)}

clock = FakeClock()
AS = MockAuthServer(clock)

# --- client side: step 1, build the authorization request
verifier, state, nonce = secrets.token_urlsafe(48), secrets.token_urlsafe(16), secrets.token_urlsafe(16)
auth_params = {"response_type": "code", "client_id": "notes-spa", "redirect_uri": "https://notes.example/callback",
               "scope": "openid email notes:read", "state": state, "nonce": nonce,
               "code_challenge": pkce_challenge(verifier), "code_challenge_method": "S256"}
print("1. browser goes to: https://auth.example/authorize?" + urlencode(auth_params)[:110] + "...")

# --- step 2, the user logs in at the AS, which redirects back with a code
resp = AS.authorize(auth_params, logged_in_user="alice")
cb = parse_qs(urlparse(resp["redirect"]).query)
print("2. callback      :", resp["redirect"][:80] + "...")

# --- step 3, the client checks state, then redeems the code with the verifier
check("3. state matches the one we stored", cb["state"][0] == state)
tokens = AS.token({"grant_type": "authorization_code", "code": cb["code"][0], "client_id": "notes-spa",
                   "redirect_uri": "https://notes.example/callback", "code_verifier": verifier})

# --- step 4, validate the ID token (who the user is) and keep the access token (for the API)
idt = jwt_verify(tokens["id_token"], AS.key, issuer=AS.ISSUER, audience="notes-spa", now=clock.now())
check("4. id_token nonce matches", idt["nonce"] == nonce)
api_claims = jwt_verify(tokens["access_token"], AS.key, issuer=AS.ISSUER, audience="notes-api", now=clock.now())
print(f"   signed in as {idt['sub']} ({idt['email']}); API scope = {api_claims['scope']!r}")

1. browser goes to: https://auth.example/authorize?response_type=code&client_id=notes-spa&redirect_uri=https%3A%2F%2Fnotes.example%2Fcallback&scope=openid+email+...
2. callback      : https://notes.example/callback?code=IEtXSDBG3ZWYLNwYFsouWslfffadWGDujN7qAYF3pYs&...
PASS 3. state matches the one we stored
PASS 4. id_token nonce matches
   signed in as alice (alice@example.com); API scope = 'openid email notes:read'


Now the attacks the flow is designed to stop:

In [20]:
def fresh_code(user="alice"):
    v = secrets.token_urlsafe(48)
    p = auth_params | {"code_challenge": pkce_challenge(v), "state": "s"}
    return parse_qs(urlparse(AS.authorize(p, user)["redirect"]).query)["code"][0], v

base = {"grant_type": "authorization_code", "client_id": "notes-spa", "redirect_uri": "https://notes.example/callback"}

code1, v1 = fresh_code()
check("intercepted code without the verifier is useless",
      "error" in AS.token(base | {"code": code1, "code_verifier": secrets.token_urlsafe(48)}))

code2, v2 = fresh_code()
AS.token(base | {"code": code2, "code_verifier": v2})
check("a code cannot be redeemed twice", "error" in AS.token(base | {"code": code2, "code_verifier": v2}))

check("redirect_uri to an attacker's site is refused",
      "error" in AS.authorize(auth_params | {"redirect_uri": "https://notes.example.evil.com/callback"}, "alice"))

check("a request without PKCE is refused", "error" in AS.authorize({k: v for k, v in auth_params.items() if not k.startswith("code_")}, "alice"))

code3, v3 = fresh_code(); clock.advance(120)
check("codes expire quickly (60 s here)", "error" in AS.token(base | {"code": code3, "code_verifier": v3}))

PASS intercepted code without the verifier is useless
PASS a code cannot be redeemed twice
PASS redirect_uri to an attacker's site is refused
PASS a request without PKCE is refused
PASS codes expire quickly (60 s here)


### 5.3 The other flows (grants)

| Grant | Who uses it | How it works (short) | Status |
|---|---|---|---|
| **Authorization code + PKCE** | any app with a user | above | use it |
| **Client credentials** | machine to machine, no user (cron jobs, backend services, AI agents acting as themselves) | the client posts its own `client_id` + secret (or a signed JWT assertion, or an mTLS cert) to `/token` and gets an access token | use it; prefer short tokens and narrow scopes |
| **Device authorization** (RFC 8628) | TVs, CLIs, devices without a good browser | device gets a `device_code` + short `user_code`; user visits a URL on their phone and types the code; device polls `/token` every `interval` seconds | use it for input-limited devices (GitHub CLI uses it) |
| **Refresh token** | any client holding a refresh token | swap the refresh token for new tokens (with rotation, section 4.4) | use it |
| **Token exchange** (RFC 8693) | a service calling another on a user's behalf; agents | trade one token for a narrower one for a different audience | use it for delegation chains |
| Implicit | old SPAs | token returned directly in the URL fragment | **deprecated**: tokens leak via history and referrers, no PKCE, no refresh |
| Resource owner password credentials | old first-party apps | the app collects the user's password and posts it to `/token` | **deprecated** (RFC 9700: MUST NOT): trains users to type passwords into apps, cannot do MFA or passkeys |

Note: the FastAPI tutorial in [12 / 02](../12_FastAPI/02_advanced_fastapi.ipynb) uses FastAPI's `OAuth2PasswordBearer` form. That is fine for a first-party demo API, but for third-party clients or real SSO, use the authorization code flow.

A client credentials request (shown, not run):

```bash
curl -s -X POST https://auth.example/oauth/token \
  -d grant_type=client_credentials -d client_id=billing-worker \
  -d client_secret="$CLIENT_SECRET" -d scope="invoices:write" -d audience=https://api.example
```

### 5.4 ID token vs access token, scopes and consent

| | ID token | Access token |
|---|---|---|
| Standard | OIDC | OAuth 2.0 |
| Audience (`aud`) | the **client** app | the **API** |
| Purpose | tell the app who logged in | let the API decide what the caller may do |
| Format | always a JWT | a JWT (RFC 9068 profile) or an opaque string |
| Common mistake | sending it to your API as a bearer token | decoding it in the frontend to decide who the user is |

**Scopes** are the permissions an access token carries (`notes:read`, `calendar.readonly`). Ask for the smallest set you need, and request more only when the user tries the feature (incremental consent). Scopes describe what the *app* may do; your API must still check what the *user* may do (section 8). OIDC's standard scopes are `openid` (required for an ID token), `profile`, `email`, `address`, `phone`, and `offline_access` (ask for a refresh token).

### 5.5 JWKS and key rotation

Providers sign tokens with a **private key** and publish the **public keys** as a JSON Web Key Set at a `jwks_uri` (found in the discovery document `/.well-known/openid-configuration`). Each token header carries a `kid` (key ID) saying which key signed it. To rotate keys: publish the new key, wait for caches to pick it up, start signing with it, and remove the old key only after every token it signed has expired.

The toy below shows the `kid` lookup and an overlap window. It uses HMAC keys to stay stdlib-only; a real JWKS contains *public* keys only.

In [21]:
class KeyRing:
    """Stand-in for a provider's signing keys + the verifier's cached JWKS."""
    def __init__(self):
        self.keys, self.active = {}, None
    def add(self, kid):
        self.keys[kid] = secrets.token_bytes(32)
    def activate(self, kid):
        self.active = kid
    def retire(self, kid):
        self.keys.pop(kid)
    def sign(self, claims):
        return jwt_sign(claims, self.keys[self.active], kid=self.active)

def verify_with_jwks(token, ring, now):
    kid = json.loads(b64url_decode(token.split(".")[0])).get("kid")
    if kid not in ring.keys:                      # real verifiers refetch the JWKS once (rate-limited) on an unknown kid
        raise JWTError(f"unknown kid {kid!r}")
    return jwt_verify(token, ring.keys[kid], issuer="https://auth.example", audience="notes-api", now=now)

ring, clock = KeyRing(), FakeClock()
ring.add("2026-10"); ring.activate("2026-10")
c = {"iss": "https://auth.example", "aud": "notes-api", "sub": "alice", "exp": clock.now() + 600}
old_token = ring.sign(c)
ring.add("2026-11")                               # 1. publish new key (both in the JWKS)
ring.activate("2026-11")                          # 2. start signing with it
new_token = ring.sign(c | {"exp": clock.now() + 600})
check("old tokens still verify during the overlap", verify_with_jwks(old_token, ring, clock.now())["sub"] == "alice")
check("new tokens verify", verify_with_jwks(new_token, ring, clock.now())["sub"] == "alice")
clock.advance(601); ring.retire("2026-10")        # 3. retire the old key after its tokens expired
expect_reject("token signed by the retired key", lambda: verify_with_jwks(old_token, ring, clock.now()))

PASS old tokens still verify during the overlap
PASS new tokens verify
PASS rejected: token signed by the retired key                  -> unknown kid '2026-10'


### 5.6 "Sign in with Google / GitHub / Microsoft / Apple" in practice

| Provider | Protocol | Notes (as of late 2026; check the docs) |
|---|---|---|
| Google | OIDC | discovery at `accounts.google.com/.well-known/openid-configuration`; use `sub` as the stable ID, not email |
| Microsoft (Entra ID, personal accounts) | OIDC | multi-tenant apps must validate the tenant (`tid`) and issuer per tenant |
| Apple | OIDC-style | users can hide their email (relay address); the user's name is sent only on the first sign-in, so store it then |
| GitHub | OAuth 2.0 (no ID token for OAuth apps) | call the user API with the access token to get the profile; no implicit grant; device flow for CLIs; expiring user tokens last 8 hours, refresh tokens 6 months |

Practical rules:
- Key accounts on **(issuer, `sub`)**, not on email. Emails change and can be reused.
- Link a social login to an existing account only when the provider says the email is **verified**, and ideally after the user proves they own the existing account.
- Register exact redirect URIs per environment (localhost, staging, prod).
- Most providers' consoles require a privacy policy URL and, for sensitive scopes, an app review. Plan for that time.

### 5.7 What OAuth 2.1 changes

OAuth 2.1 (`draft-ietf-oauth-v2-1`, at version 17 and in IESG evaluation as of late 2026, so still a draft) folds 10+ years of best practice into one document:
- PKCE required for all clients using the authorization code flow.
- Implicit and password grants removed.
- Redirect URIs compared by **exact string match**.
- No bearer tokens in URL query strings.
- Refresh tokens for public clients must be sender-constrained or one-time use (rotation).

If you follow RFC 9700 (the OAuth 2.0 Security Best Current Practice, January 2025), you are already doing OAuth 2.1. Related extras you will see: **PAR** (RFC 9126, push the authorization request to the server first), **DPoP** (RFC 9449, bind tokens to a key held by the client), and **RFC 9728** (protected resource metadata, used by MCP servers to tell clients which authorization server to use).

> **Best practices: section 5**
> - Authorization code + PKCE (S256) for every user-facing app; client credentials for machines; device flow for TVs and CLIs.
> - Validate `state` and `nonce`; exact redirect URI matching; short single-use codes.
> - Validate ID tokens fully (signature via JWKS, `iss`, `aud`, `exp`, `nonce`); never send ID tokens to APIs.
> - Small scopes, incremental consent; your API still enforces per-user permissions.
> - Use a certified library (Authlib for Python, `openid-client` for Node, Spring Security, MSAL) instead of writing the protocol yourself.

## 6. Enterprise SSO: SAML, OIDC and SCIM

**What it is:** single sign-on (SSO) means a company's employees log in to your app with their **company** account (Entra ID, Okta, Google Workspace, Ping, JumpCloud). The company's IT team controls who has access, MFA, and offboarding. Business customers usually require it before they buy.

[28 / 02 section 9.4](../28_Applied_AI_and_Forward_Deployed_Engineering/02_forward_deployed_engineering_in_practice.ipynb) walks through SSO and SCIM in a customer integration. Here is the short version plus multi-tenant routing.

### 6.1 SAML 2.0 vs OIDC

| | SAML 2.0 | OIDC |
|---|---|---|
| Age / format | 2005, XML assertions signed with XML-DSig | 2014, JSON + JWT |
| Transport | browser POSTs an XML assertion to your ACS (assertion consumer service) URL | authorization code flow (section 5) |
| Good at | enterprise web apps; every corporate IdP supports it | web, mobile, APIs; simpler to implement |
| Main risks | XML signature wrapping, accepting unsigned assertions, not checking audience/recipient/time | the OAuth pitfalls from sections 4-5 |
| Advice | **never parse SAML yourself**; use a library or a provider | prefer OIDC when the customer's IdP supports it |

### 6.2 Provisioning: SCIM and just-in-time

- **Just-in-time (JIT) provisioning:** create the user account the first time they log in through SSO, using the claims in the assertion or ID token. Simple, but the app never hears when someone *leaves*.
- **SCIM 2.0** (RFC 7643 schema, RFC 7644 protocol): the company's IdP calls *your* REST API (`POST /scim/v2/Users`, `PATCH`, `DELETE`, `/Groups`) to create, update and **deactivate** users and groups. Deprovisioning is the main reason customers ask for it: when an employee leaves, their access to your app ends automatically.

### 6.3 Multi-tenant SSO: one connection per customer

Each business customer (tenant) has its own IdP connection. Login starts with **home realm discovery**: work out which tenant from the email domain (or a tenant-specific URL such as `acme.app.example`), send the user to that tenant's IdP, and on return **check that the issuer matches the tenant you started with**, so one customer's IdP can never log people into another customer's tenant.

In [22]:
TENANTS = {
    "acme":   {"domains": {"acme.com"},   "issuer": "https://login.acme-idp.example",   "default_role": "member"},
    "globex": {"domains": {"globex.io"},  "issuer": "https://globex.okta.example",      "default_role": "viewer"},
}
USERS = {}  # (tenant, issuer, sub) -> user record

def discover_tenant(email):
    domain = email.rsplit("@", 1)[-1].lower()
    return next((t for t, cfg in TENANTS.items() if domain in cfg["domains"]), None)

def sso_callback(started_tenant, assertion):
    """assertion: already signature-checked claims from the IdP (by a SAML/OIDC library)."""
    cfg = TENANTS[started_tenant]
    if assertion["iss"] != cfg["issuer"]:
        return "rejected: issuer does not belong to this tenant"
    if assertion["email"].rsplit("@", 1)[-1] not in cfg["domains"]:
        return "rejected: email domain not owned by this tenant"
    key = (started_tenant, assertion["iss"], assertion["sub"])
    if key not in USERS:                                         # just-in-time provisioning
        USERS[key] = {"email": assertion["email"], "role": cfg["default_role"], "active": True}
    return f"logged in to {started_tenant} as {USERS[key]['email']} ({USERS[key]['role']})"

def scim_deactivate(tenant, iss, sub):
    USERS[(tenant, iss, sub)]["active"] = False                   # what a SCIM PATCH active=false does

print("discover:", discover_tenant("bob@acme.com"))
print(sso_callback("acme", {"iss": "https://login.acme-idp.example", "sub": "u-77", "email": "bob@acme.com"}))
print(sso_callback("acme", {"iss": "https://globex.okta.example", "sub": "u-1", "email": "mallory@globex.io"}))
scim_deactivate("acme", "https://login.acme-idp.example", "u-77")
print("after SCIM deactivate, active =", USERS[("acme", "https://login.acme-idp.example", "u-77")]["active"])

discover: acme
logged in to acme as bob@acme.com (member)
rejected: issuer does not belong to this tenant
after SCIM deactivate, active = False


**What you can use:** WorkOS, Auth0 (Okta) Enterprise Connections, Clerk, Stytch (now part of Twilio), Descope, FusionAuth, Frontegg, Keycloak (identity brokering), Ory Polis (formerly BoxyHQ Jackson), or the cloud IdPs in section 11. These hand you one OIDC-style integration and handle each customer's SAML quirks.

> **Best practices: section 6**
> - Offer OIDC and SAML; do not write your own SAML parser.
> - Key users by (tenant, issuer, subject); enforce issuer-to-tenant and domain-to-tenant mapping; verify domain ownership (DNS TXT record) before a tenant can claim it.
> - Support SCIM deprovisioning, or at minimum short sessions so removed employees lose access quickly.
> - Let customers *require* SSO for their domain, and keep a break-glass admin login for when their IdP is down.

## 7. API authentication

### 7.1 API keys

**What it is:** a long random string a developer puts in a header (`Authorization: Bearer ...` or `X-API-Key`). Simple and right for server-to-server calls by your customers. [16 / 06 section 3.3](../16_Deployment/06_deploying_an_ai_saas_application_end_to_end.ipynb) uses hashed keys per tenant; here are the production details:

- **Prefix** that says what it is: `zta_live_` vs `zta_test_`. Prefixes let secret scanners (GitHub push protection and similar) recognise and report leaked keys, and let humans see at a glance which environment a key is for.
- **Key ID + secret**: the ID part lets you find the row quickly; the secret part is checked against a **hash**. Because the secret is 256 random bits, a fast SHA-256 is fine (no slow hash needed).
- An optional **checksum** at the end lets scanners and your API reject typos and fakes without a database lookup (GitHub's token format does this).
- **Scopes**, an owner, an expiry, `last_used_at`, and a **per-key rate limit**.
- **Rotation without downtime**: create a second key, deploy it, then revoke the first. Allow at least two active keys per client.
- Show the full key **once** at creation.

In [23]:
import zlib

class APIKeyStore:
    def __init__(self, clock):
        self.clock, self.rows, self.buckets = clock, {}, {}

    def create(self, owner, scopes, env="live", rate_per_min=60):
        alnum = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"   # no '_' so the key splits cleanly
        key_id, secret = secrets.token_hex(6), "".join(secrets.choice(alnum) for _ in range(43))  # ~256 bits
        body = f"zta_{env}_{key_id}_{secret}"
        full = f"{body}_{zlib.crc32(body.encode()):08x}"            # checksum suffix
        self.rows[key_id] = {"hash": hashlib.sha256(secret.encode()).hexdigest(), "owner": owner, "scopes": set(scopes),
                             "rate": rate_per_min, "revoked": False, "last_used": None}
        return full                                                # show once, never store in plain text

    def authenticate(self, presented, needed_scope):
        try:
            prefix, env, key_id, secret, crc = presented.split("_")
        except ValueError:
            return 401, "malformed key"
        if f"{zlib.crc32(f'{prefix}_{env}_{key_id}_{secret}'.encode()):08x}" != crc:
            return 401, "bad checksum (typo or fake), no DB lookup needed"
        row = self.rows.get(key_id)
        if not row or row["revoked"] or not hmac.compare_digest(row["hash"], hashlib.sha256(secret.encode()).hexdigest()):
            return 401, "invalid key"
        if needed_scope not in row["scopes"]:
            return 403, f"key lacks scope {needed_scope}"
        tokens, last = self.buckets.get(key_id, (row["rate"], self.clock.now()))   # token bucket per key
        tokens = min(row["rate"], tokens + (self.clock.now() - last) * row["rate"] / 60)
        if tokens < 1:
            self.buckets[key_id] = (tokens, self.clock.now())
            return 429, "rate limited (per key)"
        self.buckets[key_id] = (tokens - 1, self.clock.now())
        row["last_used"] = self.clock.now()
        return 200, f"ok, owner={row['owner']}"

clock = FakeClock()
keys = APIKeyStore(clock)
k1 = keys.create("acme", {"chat:write"}, rate_per_min=3)
print("new key (shown once):", k1)
print(keys.authenticate(k1, "chat:write"))
typo = k1[:-1] + ("0" if k1[-1] != "0" else "1")
print(keys.authenticate(typo, "chat:write"))
print(keys.authenticate(k1, "admin:delete"))
print([keys.authenticate(k1, "chat:write")[0] for _ in range(4)], "<- burst beyond 3/min gets 429")

k2 = keys.create("acme", {"chat:write"})                     # rotation: second key goes live first
keys.rows[k1.split("_")[2]]["revoked"] = True                # then the old one is revoked
print("old key after rotation:", keys.authenticate(k1, "chat:write"), "| new key:", keys.authenticate(k2, "chat:write"))

new key (shown once): zta_live_c4f0b8612279_1PkuaE0bMFbZqdvbKc2agnbMSuqOS9M1ukdsvkwbjey_a0d0876c
(200, 'ok, owner=acme')
(401, 'bad checksum (typo or fake), no DB lookup needed')
(403, 'key lacks scope admin:delete')
[200, 200, 429, 429] <- burst beyond 3/min gets 429
old key after rotation: (401, 'invalid key') | new key: (200, 'ok, owner=acme')


### 7.2 HMAC request signing (the AWS SigV4 idea)

**What it is:** instead of sending the secret, the client **signs each request** with it. The server recomputes the signature. A stolen request cannot be changed (the signature covers method, path, query, body hash and time) and cannot be replayed later (the timestamp must be fresh, and a nonce cache rejects exact repeats). AWS Signature Version 4 works this way (with extra steps such as a derived signing key per date, region and service), and so do many payment and webhook APIs.

**How it works:**
1. Build a **canonical request**: `METHOD \n path \n sorted query \n sha256(body) \n timestamp \n nonce`.
2. `signature = HMAC-SHA256(secret, canonical request)`.
3. Send `key_id`, `timestamp`, `nonce` and `signature` in headers.
4. The server looks up the secret by `key_id`, checks the timestamp is within a window (5 minutes here), checks the nonce is new, recomputes, and compares in constant time.

In [24]:
def canonical(method, path, query: dict, body: bytes, ts: int, nonce: str) -> bytes:
    q = "&".join(f"{k}={query[k]}" for k in sorted(query))
    return "\n".join([method.upper(), path, q, hashlib.sha256(body).hexdigest(), str(ts), nonce]).encode()

def sign_request(key_id, secret, method, path, query, body, ts):
    nonce = secrets.token_hex(12)
    sig = hmac.new(secret, canonical(method, path, query, body, ts, nonce), hashlib.sha256).hexdigest()
    return {"X-Key-Id": key_id, "X-Timestamp": str(ts), "X-Nonce": nonce, "X-Signature": sig}

class SignedRequestVerifier:
    def __init__(self, clock, secrets_by_id, window=300):
        self.clock, self.secrets, self.window, self.seen = clock, secrets_by_id, window, {}
    def verify(self, method, path, query, body, headers):
        secret = self.secrets.get(headers.get("X-Key-Id"))
        if secret is None: return "unknown key"
        ts = int(headers["X-Timestamp"])
        if abs(self.clock.now() - ts) > self.window: return "stale timestamp"
        self.seen = {n: t for n, t in self.seen.items() if self.clock.now() - t <= self.window}   # forget old nonces
        if headers["X-Nonce"] in self.seen: return "replay (nonce seen)"
        expected = hmac.new(secret, canonical(method, path, query, body, ts, headers["X-Nonce"]), hashlib.sha256).hexdigest()
        if not hmac.compare_digest(expected, headers["X-Signature"]): return "bad signature"
        self.seen[headers["X-Nonce"]] = ts
        return "ok"

clock = FakeClock()
shared = {"partner-7": secrets.token_bytes(32)}
srv = SignedRequestVerifier(clock, shared)
body = json.dumps({"amount": 100, "to": "acct_1"}).encode()
hdrs = sign_request("partner-7", shared["partner-7"], "POST", "/v1/transfers", {"dry_run": "false"}, body, clock.now())

check("signed request verifies", srv.verify("POST", "/v1/transfers", {"dry_run": "false"}, body, hdrs) == "ok")
check("exact replay is rejected", srv.verify("POST", "/v1/transfers", {"dry_run": "false"}, body, hdrs).startswith("replay"))
tampered = json.dumps({"amount": 100000, "to": "acct_666"}).encode()
h2 = sign_request("partner-7", shared["partner-7"], "POST", "/v1/transfers", {"dry_run": "false"}, body, clock.now())
check("tampered body is rejected", srv.verify("POST", "/v1/transfers", {"dry_run": "false"}, tampered, h2) == "bad signature")
h3 = sign_request("partner-7", shared["partner-7"], "POST", "/v1/transfers", {"dry_run": "false"}, body, clock.now())
clock.advance(600)
check("a request captured 10 minutes ago is rejected", srv.verify("POST", "/v1/transfers", {"dry_run": "false"}, body, h3) == "stale timestamp")

PASS signed request verifies
PASS exact replay is rejected
PASS tampered body is rejected
PASS a request captured 10 minutes ago is rejected


### 7.3 Webhook signatures

Webhooks are the same idea in reverse: *you* receive signed requests from a provider (Stripe, GitHub, your billing system). Verify the HMAC over the **raw body bytes** (before JSON parsing), check the timestamp, and de-duplicate by event ID. The **Standard Webhooks** specification (`webhook-id`, `webhook-timestamp`, `webhook-signature` headers) is a common format. Both [28 / 02 section 9.1](../28_Applied_AI_and_Forward_Deployed_Engineering/02_forward_deployed_engineering_in_practice.ipynb) and [16 / 06 section 3.10](../16_Deployment/06_deploying_an_ai_saas_application_end_to_end.ipynb) have runnable verifiers.

### 7.4 mTLS, service-to-service auth and workload identity

| Technique | What it is | When to use |
|---|---|---|
| **mTLS** | both sides present certificates during the TLS handshake, so the server knows exactly which client is connecting | internal service meshes (Istio, Linkerd issue and rotate certs automatically), banking/open-banking APIs, IoT. RFC 8705 can bind OAuth tokens to the client certificate |
| **Client credentials JWTs** | services get short-lived access tokens from an internal AS (section 5.3) | when services already speak OAuth |
| **Workload identity** | the platform gives each workload an identity it can prove without stored secrets: a cloud IAM role (AWS IAM roles for EKS pods, Azure managed identities, GCP service accounts), or a SPIFFE ID (SPIRE) | anything running in the cloud or Kubernetes: **no long-lived keys in env vars** |
| **OIDC federation** | an external system's signed OIDC token (for example from GitHub Actions) is exchanged for short-lived cloud credentials | CI/CD deploying to the cloud: see [20 / 07 section 10](../20_Git_and_DevOps/07_devsecops_cicd_supply_chain_and_platform_engineering.ipynb) |

**AI agents and tools:** an agent should get its **own** identity and short-lived, narrowly scoped tokens per tool (often via token exchange), not the user's full session. [10 / 16 section 7](../10_Agents/16_agentic_workflow_patterns_and_deep_research.ipynb) has a toy token gateway, and the MCP authorization spec builds on OAuth 2.1, PKCE and protected resource metadata (RFC 9728); see [10 / 05](../10_Agents/05_mcp_and_tools.ipynb).

> **Best practices: section 7**
> - API keys: prefixed, high-entropy, hashed at rest, scoped, expiring, rate limited per key, rotatable with overlap.
> - Signed requests and webhooks: sign the raw bytes, check freshness and nonces/event IDs, compare in constant time.
> - Inside your platform, prefer workload identity and mTLS over shared secrets; never put cloud keys in containers.
> - Log the key ID (never the key) on every request so you can trace and revoke.

## 8. Authorization models

Authentication gets the user in the door. **Authorization decides every single action.** Broken access control is number 1 in the OWASP Top 10, and the usual bug is simple: the server checks *that* you are logged in, but not *whether this object is yours* (an **IDOR**, insecure direct object reference).

### 8.1 The three models

| Model | Rule shape | Analogy | Good for | Weak at |
|---|---|---|---|---|
| **RBAC** (role-based) | user has role, role has permissions | job titles: "editors can edit" | admin panels, internal tools, coarse plans | per-object sharing ("Bob may see *this* doc"): role explosion |
| **ABAC** (attribute-based) | rule over attributes of user, resource, action, context | "staff with clearance >= secret, on the office network, during work hours" | data classification, regions, time, device posture | hard to answer "who can see X?" |
| **ReBAC** (relationship-based) | permission follows relationships in a graph | "you can see the doc because you are in a team that owns the folder that contains it" | sharing, folders, orgs, Google-Drive-style apps | needs a tuple store and a graph walk |

**ReBAC and Google Zanzibar.** Google described Zanzibar (USENIX ATC 2019), the system behind sharing in Drive, YouTube and more. Permissions are stored as **relation tuples** like `doc:7#viewer@user:bob` or `doc:7#viewer@team:eng#member`, and a check walks the graph. Open-source systems inspired by it: **OpenFGA** (CNCF incubating since October 2025), **SpiceDB** (AuthZed), **Permify** (now part of FusionAuth) and **Ory Keto**.

### 8.2 A mini RBAC + ABAC + ReBAC checker for a document-sharing app

In [25]:
# --- toy data ------------------------------------------------------------
USERS_DB = {
    "alice": {"org": "acme",   "role": "admin",  "clearance": 2},
    "bob":   {"org": "acme",   "role": "member", "clearance": 1},
    "carol": {"org": "acme",   "role": "member", "clearance": 2},
    "dave":  {"org": "globex", "role": "admin",  "clearance": 2},   # admin of ANOTHER tenant
}
DOCS = {
    1: {"org": "acme",   "title": "Roadmap",         "classification": 1, "folder": "f-eng"},
    2: {"org": "acme",   "title": "Salaries",        "classification": 2, "folder": "f-hr"},
    3: {"org": "globex", "title": "Globex secrets",  "classification": 1, "folder": "f-gx"},
}
# ReBAC tuples: (object, relation, subject)
TUPLES = {
    ("folder:f-eng", "viewer", "team:eng#member"),
    ("team:eng", "member", "user:bob"),
    ("doc:1", "parent", "folder:f-eng"),
    ("doc:2", "owner", "user:carol"),
    ("doc:2", "parent", "folder:f-hr"),
}

# --- RBAC: role -> permissions (within your own org only) -----------------
ROLE_PERMS = {"admin": {"read", "edit", "delete", "share"}, "member": set()}   # members get access via sharing

# --- ReBAC: owner implies editor implies viewer; viewers of a parent folder can view children
IMPLIED = {"viewer": ["viewer", "editor", "owner"], "editor": ["editor", "owner"], "owner": ["owner"]}

def rebac_has(obj, relation, user, depth=0):
    if depth > 5: return False                                    # guard against cycles
    for rel in IMPLIED[relation]:
        for (o, r, s) in TUPLES:
            if o != obj or r != rel: continue
            if s == f"user:{user}": return True
            if "#" in s:                                          # userset, e.g. team:eng#member
                group, grel = s.split("#")
                if (group, grel, f"user:{user}") in TUPLES: return True
    for (o, r, s) in TUPLES:                                      # inherit from parent folder
        if o == obj and r == "parent" and rebac_has(s, relation, user, depth + 1): return True
    return False

ACTION_TO_RELATION = {"read": "viewer", "edit": "editor", "delete": "owner", "share": "owner"}

def can(user_id, action, doc_id, context=None) -> tuple:
    """Deny by default. Returns (allowed, reason)."""
    user, doc = USERS_DB.get(user_id), DOCS.get(doc_id)
    if user is None or doc is None:
        return False, "unknown user or doc"
    if user["org"] != doc["org"]:                                 # tenant isolation first, always
        return False, "different tenant"
    # ABAC: classification needs clearance, and level-2 docs only from a managed device
    if user["clearance"] < doc["classification"]:
        return False, "clearance too low"
    if doc["classification"] >= 2 and not (context or {}).get("managed_device"):
        return False, "level-2 docs need a managed device"
    if action in ROLE_PERMS[user["role"]]:                        # RBAC
        return True, f"role {user['role']}"
    if rebac_has(f"doc:{doc_id}", ACTION_TO_RELATION[action], user_id):   # ReBAC
        return True, "relationship"
    return False, "no grant (default deny)"

for args in [("bob", "read", 1, None), ("bob", "edit", 1, None), ("bob", "read", 2, None),
             ("carol", "read", 2, {"managed_device": True}), ("carol", "read", 2, {"managed_device": False}),
             ("alice", "delete", 1, None), ("dave", "read", 1, None)]:
    print(f"can{args[:3]!s:28} ctx={args[3]!s:26} -> {can(*args)}")

can('bob', 'read', 1)           ctx=None                       -> (True, 'relationship')
can('bob', 'edit', 1)           ctx=None                       -> (False, 'no grant (default deny)')
can('bob', 'read', 2)           ctx=None                       -> (False, 'clearance too low')
can('carol', 'read', 2)         ctx={'managed_device': True}   -> (True, 'relationship')
can('carol', 'read', 2)         ctx={'managed_device': False}  -> (False, 'level-2 docs need a managed device')
can('alice', 'delete', 1)       ctx=None                       -> (True, 'role admin')
can('dave', 'read', 1)          ctx=None                       -> (False, 'different tenant')


### 8.3 Check on every request, server side, deny by default: tests that catch an IDOR

Two versions of the same endpoint. Version 1 only checks that the user is logged in. The tests below are the kind you should have in CI for every endpoint that takes an object ID.

In [26]:
def get_document_v1(session_user, doc_id):
    """VULNERABLE: authenticated, but no object-level authorization."""
    if session_user is None:
        return 401, None
    doc = DOCS.get(doc_id)
    return (200, doc) if doc else (404, None)

def get_document_v2(session_user, doc_id, context=None):
    """FIXED: checks the object on every request; 404 hides other tenants' docs."""
    if session_user is None:
        return 401, None
    allowed, reason = can(session_user, "read", doc_id, context)
    if not allowed:
        return (404, None) if reason in ("different tenant", "unknown user or doc") else (403, None)
    return 200, DOCS[doc_id]

def authz_test_suite(endpoint):
    cases = [  # (user, doc, expected status, why)
        (None,    1, 401, "anonymous"),
        ("bob",   1, 200, "shared via team folder"),
        ("bob",   2, 403, "same tenant, not shared"),
        ("dave",  1, 404, "other tenant admin must not see acme docs"),
        ("bob",   3, 404, "IDOR: bob changes the id to a globex doc"),
        ("bob", 999, 404, "missing doc"),
    ]
    failures = 0
    for user, doc, expected, why in cases:
        status, _ = endpoint(user, doc)
        ok = status == expected
        failures += not ok
        print(f"  {'PASS' if ok else 'FAIL'} {why:<45} expected {expected}, got {status}")
    return failures

print("v1 (only checks login):"); f1 = authz_test_suite(get_document_v1)
print("v2 (checks the object):"); f2 = authz_test_suite(get_document_v2)
check("the test suite catches the IDOR in v1", f1 > 0)
check("v2 passes every authorization test", f2 == 0)

v1 (only checks login):
  PASS anonymous                                     expected 401, got 401
  PASS shared via team folder                        expected 200, got 200
  FAIL same tenant, not shared                       expected 403, got 200
  FAIL other tenant admin must not see acme docs     expected 404, got 200
  FAIL IDOR: bob changes the id to a globex doc      expected 404, got 200
  PASS missing doc                                   expected 404, got 404
v2 (checks the object):
  PASS anonymous                                     expected 401, got 401
  PASS shared via team folder                        expected 200, got 200
  PASS same tenant, not shared                       expected 403, got 403
  PASS other tenant admin must not see acme docs     expected 404, got 404
  PASS IDOR: bob changes the id to a globex doc      expected 404, got 404
  PASS missing doc                                   expected 404, got 404
PASS the test suite catches the IDOR in v1
PASS v2 p

### 8.4 Policy engines and where the check lives

Hard-coding `if` statements works for small apps. As rules grow, teams move them into a **policy engine**: your code asks "may user U do action A on resource R?" and the engine answers, from rules stored and tested separately.

| Tool | Model | Language / style | Notes |
|---|---|---|---|
| **OPA** (Open Policy Agent) | general ABAC/RBAC | Rego | CNCF graduated (2021); also used for Kubernetes admission and infrastructure policies |
| **Cedar** | RBAC + ABAC | Cedar language, built to be analysable | from AWS; the engine behind Amazon Verified Permissions; open source |
| **Casbin** | ACL, RBAC, ABAC | model + policy files | a library in many languages; good for embedding |
| **OpenFGA** | ReBAC (Zanzibar) | authorization model + tuples | CNCF incubating; originally from Auth0/Okta |
| **SpiceDB** | ReBAC (Zanzibar) | schema + relationships | AuthZed; strong consistency options |
| **Permify**, **Ory Keto** | ReBAC (Zanzibar) | schema + tuples | open source |
| Provider features | RBAC, organizations | dashboard + token claims | Auth0, Clerk, WorkOS, Keycloak, Entra app roles: fine for coarse roles |

A Cedar policy and a Rego rule, for flavour (shown, not run):

```text
// Cedar
permit (principal, action == Action::"read", resource)
when { resource.org == principal.org && principal.clearance >= resource.classification };
```

```text
# Rego (OPA)
package docs
default allow := false
allow if {
    input.user.org == input.doc.org
    input.user.clearance >= input.doc.classification
    input.action == "read"
}
```

### 8.5 Multi-tenant authorization in the database

Defence in depth: even if an endpoint forgets a check, the database should refuse to return another tenant's rows. PostgreSQL **row-level security** (RLS) does this, and Supabase builds its whole client-side access model on it. [27 / 02 section 5](../27_AI_and_Cybersecurity/02_securing_vibe_coded_applications.ipynb) emulates RLS in SQLite; [16 / 06 section 2.3](../16_Deployment/06_deploying_an_ai_saas_application_end_to_end.ipynb) covers tenant isolation patterns.

```sql
ALTER TABLE documents ENABLE ROW LEVEL SECURITY;
CREATE POLICY tenant_isolation ON documents
  USING (org_id = current_setting('app.current_org')::uuid);
-- the API sets this per request, inside the transaction:
SET LOCAL app.current_org = '...tenant id from the verified token...';
```

**Limits to know:** putting every permission into the token does not scale (token size, section 4.3, and stale permissions until the token expires). Keep roles coarse in tokens and check fine-grained permissions at request time. ReBAC systems answer "check" fast but "list everything Bob can see" is harder; plan for it (OpenFGA's `ListObjects`, SpiceDB's `LookupResources`).

> **Best practices: section 8**
> - Deny by default; check on every request, on the server, at the object level; tenant check first.
> - Return 404 for objects in other tenants so IDs cannot be probed.
> - Write authorization tests per endpoint (like the suite above) and run them in CI.
> - Centralise rules (a `can()` function or a policy engine) instead of scattering `if`s.
> - Add database-level isolation (RLS or per-tenant schemas) as a second wall.

## 9. Browser security around auth

### 9.1 CORS: what it does and does not protect

**What it is:** the browser's **same-origin policy** stops JavaScript on `evil.example` from *reading* responses from `api.example`. **CORS** (Cross-Origin Resource Sharing) is how your server *relaxes* that rule for origins you trust, using `Access-Control-Allow-*` headers.

**Simple analogy:** CORS is the browser asking your server "may this other website read your reply?" It is a rule the *browser* follows on behalf of the user. It is not a lock on your API.

| CORS does | CORS does NOT |
|---|---|
| stop other sites' scripts from **reading** your API's responses in a victim's browser | stop `curl`, scripts, servers or bots from calling your API (they ignore CORS) |
| make the browser send a **preflight** `OPTIONS` before "non-simple" requests (JSON body, custom headers, PUT/DELETE) | stop "simple" requests (a form POST) from being **sent**: that is CSRF, handled in 9.2 |
| refuse to expose credentialed responses unless the origin is listed exactly | replace authentication or authorization |

[23 / 10 section 7.8](10_connecting_ai_to_a_frontend.ipynb) tests CORS on a real FastAPI app, and [27 / 02 section 3.10](../27_AI_and_Cybersecurity/02_securing_vibe_coded_applications.ipynb) shows the "trust every origin" mistake. Here is a small simulator of the browser's decision.

In [27]:
SAFE_HEADERS = {"accept", "accept-language", "content-language", "content-type"}
SIMPLE_TYPES = {"application/x-www-form-urlencoded", "multipart/form-data", "text/plain"}

def is_simple(method, headers):
    if method not in ("GET", "HEAD", "POST"): return False
    if any(h.lower() not in SAFE_HEADERS for h in headers): return False
    return headers.get("Content-Type", "text/plain").split(";")[0] in SIMPLE_TYPES

def browser_cors(page_origin, api_origin, method, headers, with_credentials, server):
    """Return what the browser does. `server` is the API's CORS config."""
    if page_origin == api_origin:
        return "same origin: no CORS needed"
    origin_ok = page_origin in server["allow_origins"] or ("*" in server["allow_origins"] and not with_credentials)
    if not is_simple(method, headers):
        pre_ok = (origin_ok and method in server["allow_methods"]
                  and all(h.lower() in SAFE_HEADERS or h.lower() in server["allow_headers"] for h in headers)
                  and (server["allow_credentials"] or not with_credentials))
        if not pre_ok:
            return "preflight FAILED: request never sent"
        sent = f"preflight ok (cached {server['max_age']}s), request sent"
    else:
        sent = "simple request: SENT without preflight (any server-side effect still happens)"
    readable = origin_ok and (server["allow_credentials"] or not with_credentials)
    return sent + (" | JS can read the response" if readable else " | browser HIDES the response from JS")

server = {"allow_origins": {"https://app.example"}, "allow_methods": {"GET", "POST", "PUT"},
          "allow_headers": {"authorization", "x-csrf-token"}, "allow_credentials": True, "max_age": 600}
API = "https://api.example"
cases = [
    ("https://app.example",  "PUT",  {"Content-Type": "application/json"}, True),
    ("https://evil.example", "PUT",  {"Content-Type": "application/json"}, True),
    ("https://evil.example", "POST", {"Content-Type": "application/x-www-form-urlencoded"}, True),
    ("https://evil.example", "GET",  {}, True),
    ("https://app.example",  "DELETE", {"Authorization": "Bearer x"}, True),   # DELETE is not in allow_methods
]
for origin, m, h, cred in cases:
    print(f"{origin:<22} {m:<6} {list(h.values()) or '-'!s:38} -> {browser_cors(origin, API, m, h, cred, server)}")

https://app.example    PUT    ['application/json']                   -> preflight ok (cached 600s), request sent | JS can read the response
https://evil.example   PUT    ['application/json']                   -> preflight FAILED: request never sent
https://evil.example   POST   ['application/x-www-form-urlencoded']  -> simple request: SENT without preflight (any server-side effect still happens) | browser HIDES the response from JS
https://evil.example   GET    -                                      -> simple request: SENT without preflight (any server-side effect still happens) | browser HIDES the response from JS
https://app.example    DELETE ['Bearer x']                           -> preflight FAILED: request never sent


Read row 3 carefully: a cross-site **form POST** is a "simple" request, so the browser **sends it with the user's cookies** even though CORS hides the response. If that request changes something (transfer money, change email), CORS did nothing to stop it. That is CSRF.

### 9.2 CSRF: tokens, SameSite and Fetch Metadata

**What it is:** cross-site request forgery makes the victim's browser send a state-changing request to your site, riding on the victim's cookies.

Defences (OWASP CSRF Prevention Cheat Sheet), use more than one:
1. **`SameSite=Lax` or `Strict` cookies**: the browser does not attach the cookie to cross-site POSTs. A strong baseline, but not complete (same-*site* subdomains, old browsers, GET endpoints that change state).
2. **CSRF tokens**: the synchronizer token pattern (a random token stored in the session and echoed in a hidden form field or header), or the **signed double-submit cookie** (an HMAC token tied to the session). OWASP discourages the *naive* unsigned double-submit pattern.
3. **Fetch Metadata**: modern browsers send `Sec-Fetch-Site: same-origin | same-site | cross-site | none`. Reject state-changing requests marked `cross-site`, with an `Origin` header check as the fallback.
4. Never change state on `GET`.

APIs that use `Authorization: Bearer` headers (not cookies) are not exposed to CSRF, because browsers never add that header automatically.

In [28]:
CSRF_KEY = secrets.token_bytes(32)

def csrf_token_for(session_id):                       # signed double-submit token tied to the session
    msg = f"{session_id}!{secrets.token_hex(8)}"
    return msg + "." + hmac.new(CSRF_KEY, msg.encode(), hashlib.sha256).hexdigest()

def csrf_ok(session_id, token):
    try:
        msg, sig = token.rsplit(".", 1)
    except (ValueError, AttributeError):
        return False
    good_sig = hmac.compare_digest(sig, hmac.new(CSRF_KEY, msg.encode(), hashlib.sha256).hexdigest())
    return good_sig and msg.split("!")[0] == session_id

def allow_state_change(method, headers, session_id, our_origin="https://app.example"):
    if method in ("GET", "HEAD", "OPTIONS"):
        return True, "safe method"
    site = headers.get("Sec-Fetch-Site")
    if site == "cross-site":
        return False, "Fetch Metadata: cross-site"
    if site is None and headers.get("Origin") not in (None, our_origin):     # fallback for old browsers
        return False, "Origin mismatch"
    if not csrf_ok(session_id, headers.get("X-CSRF-Token")):
        return False, "missing or bad CSRF token"
    return True, "ok"

sid = "sess-abc"
tok = csrf_token_for(sid)
print(allow_state_change("POST", {"Sec-Fetch-Site": "same-origin", "X-CSRF-Token": tok}, sid))
print(allow_state_change("POST", {"Sec-Fetch-Site": "cross-site"}, sid))
print(allow_state_change("POST", {"Origin": "https://evil.example"}, sid))
print(allow_state_change("POST", {"Sec-Fetch-Site": "same-site", "X-CSRF-Token": csrf_token_for("other-session")}, sid))

(True, 'ok')
(False, 'Fetch Metadata: cross-site')
(False, 'Origin mismatch')
(False, 'missing or bad CSRF token')


### 9.3 XSS and token theft, CSP and clickjacking

- **XSS** (cross-site scripting): attacker-controlled text runs as JavaScript in your page. Once that happens, the script can do anything the user can, and it can **read** any token in `localStorage`. HttpOnly cookies cannot be read, though the script can still *use* the session while the page is open, which is why short sessions and step-up re-authentication matter. Main fix: encode output (frameworks like React do this by default; avoid `dangerouslySetInnerHTML` / `innerHTML` with untrusted data, including LLM output). See [23 / 08 section 2.4](08_security_for_ai_applications.ipynb).
- **Content Security Policy (CSP):** a response header that tells the browser which scripts may run. A strict policy blocks most injected scripts even when an XSS bug exists. Example: `Content-Security-Policy: default-src 'self'; script-src 'self' 'nonce-<random per response>'; object-src 'none'; base-uri 'none'; frame-ancestors 'none'`. Roll out with `Content-Security-Policy-Report-Only` first.
- **Clickjacking:** a malicious site loads your page in an invisible `<iframe>` and tricks the user into clicking "Approve". Stop it with `frame-ancestors 'none'` (or `'self'`) in CSP, plus `X-Frame-Options: DENY` for old browsers. Login, consent and payment pages must never be framable.

Other headers worth setting on auth pages: `Strict-Transport-Security` (HSTS), `Referrer-Policy: no-referrer` (keeps reset tokens out of referrers), `Cache-Control: no-store` on pages that show tokens or personal data.

> **Best practices: section 9**
> - CORS: an explicit allow-list of origins, never reflect any `Origin`, never `*` with credentials. Remember it protects *reads in browsers* only.
> - CSRF: SameSite cookies + a CSRF token or Fetch Metadata check on every state-changing route.
> - XSS: output encoding, a strict nonce-based CSP, HttpOnly cookies for sessions.
> - `frame-ancestors 'none'` on login and consent pages; HSTS everywhere.

## 10. Choosing an auth provider vs building your own

**Build or buy?** Building login is easy; building *all of it* (password reset, MFA, passkeys, social login, SSO per customer, SCIM, bot protection, breach detection, audit logs, account recovery, compliance) is a long project with a high cost of mistakes. Most teams buy or self-host a proven product and spend their time on authorization, which is specific to their app and cannot be outsourced.

### 10.1 The options (as of late 2026; features and plans change, check the docs)

| Product | Type | Strengths | Think about |
|---|---|---|---|
| **Auth0** (part of Okta) | hosted CIAM | very complete: Universal Login, Actions (hooks), enterprise connections, Organizations for B2B | cost grows with users and features; rate limits per tenant (token-bucket) |
| **Okta Workforce Identity** | hosted workforce IdP | employee SSO, lifecycle management | for your staff, not usually for your app's customers |
| **Clerk** | hosted, UI components | fastest for React/Next.js; prebuilt sign-in UI, organizations, passkeys; very short (60 s) session JWTs refreshed by its SDK | front-end centred; vendor lock-in of the UI |
| **Supabase Auth** | hosted or self-hosted (open source) | part of Supabase; JWTs work directly with Postgres RLS | built-in email sender limited to a few emails/hour: add your own SMTP |
| **Firebase Authentication** | hosted (Google) | mobile SDKs, anonymous auth, phone auth | upgrade to Identity Platform for SAML/OIDC, multi-tenancy and SLAs; daily limits on the free plan |
| **Amazon Cognito** (user pools) | hosted (AWS) | AWS integration (API Gateway, ALB, IAM via identity pools); feature plans Lite, Essentials, Plus | request-rate quotas per API category per Region; customisation can be awkward |
| **Microsoft Entra External ID** | hosted (Azure) | customer identity on the Entra platform; successor to Azure AD B2C (no longer sold to new customers since 1 May 2025) | best fit for Microsoft-centric stacks |
| **Keycloak** | self-hosted, open source (CNCF incubating) | full OIDC/SAML server, identity brokering, user federation (LDAP), fine-grained admin | you run, patch, scale and back it up |
| **Ory** (Kratos, Hydra, Keto, Polis) | open source + Ory Network (hosted) | API-first building blocks: identity, OAuth2/OIDC server, Zanzibar permissions, enterprise SSO | more assembly than a drop-in widget |
| **authentik** | self-hosted, open source | IdP with SAML, OIDC, SCIM, proxy auth for internal apps; popular for homelabs and internal tools | you operate it |
| **WorkOS** | hosted, B2B add-on | enterprise SSO, Directory Sync (SCIM), admin portal for your customers' IT teams; AuthKit for full auth | adds enterprise features to whatever you already use |
| **Better Auth** | library (TypeScript, open source) | auth inside your own app and database; plugins for 2FA, passkeys, organizations, SSO | you own the database and operations |
| **Auth.js** (formerly NextAuth.js) | library (TypeScript) | widely used; since September 2025 maintained by the Better Auth team, who recommend Better Auth for new projects | security fixes continue; new features go to Better Auth |
| **Lucia** | *deprecated* (March 2025) | now a learning resource: a guide and copy-paste session code | do not start new projects on the npm package |
| Python frameworks | libraries | Django's built-in auth + `django-allauth`; FastAPI with `fastapi-users` or Authlib; Flask-Login | you build the flows on top |

Others you will meet: **Stytch** (now part of Twilio), **Descope**, **FusionAuth**, **Zitadel**, **Frontegg**, **Kinde**, **PropelAuth**, **SuperTokens**.

### 10.2 A decision table

| Your situation | A sensible starting point |
|---|---|
| Next.js/React product, small team, want UI done for you | Clerk, Auth0, or Better Auth (if you want it in your own DB) |
| Already on Supabase or Firebase | Supabase Auth / Firebase Auth (Identity Platform when you need SSO) |
| All-in on one cloud | Cognito (AWS), Entra External ID (Azure), Identity Platform (GCP) |
| B2B SaaS that must sell to enterprises | your auth of choice + WorkOS (or Auth0 Organizations, Clerk, Stytch, Descope) for SSO/SCIM |
| Data must stay on your servers, or on-premises / air-gapped | Keycloak, Ory (self-hosted), authentik, Zitadel |
| Internal tools for employees | your company IdP (Entra ID, Okta, Google Workspace) + an identity-aware proxy |
| Complex sharing / permissions | any of the above for authN + OpenFGA, SpiceDB, Cedar or OPA for authZ |
| Learning, or a tiny side project | a framework's built-in auth, carefully, with the rules in this notebook |

### 10.3 Limits you will run into, and how to work within them

| Limit | Typical value (examples, check your provider) | How to work within it |
|---|---|---|
| Cookie size | about 4096 bytes per cookie, ~50 cookies per domain (RFC 6265 minimums) | store an ID, not data; split or move big tokens server-side (BFF) |
| Token / header size | proxies often cap all headers at 8-16 KB; big JWTs with many groups break this | few claims; look up groups server-side; Entra ID, for example, switches to a "groups overage" claim when a user is in too many groups |
| Token lifetimes | Clerk session JWT 60 s; GitHub user tokens 8 h, refresh 6 months; Cognito refresh 60 min to 10 years (default 30 days) | short access tokens + rotated refresh tokens; handle 401 by refreshing once, then re-login |
| Auth API rate limits | Auth0: token-bucket per endpoint; Cognito: requests per second per API category, per Region; Firebase: 100 new accounts per hour per IP | cache JWKS and user profiles; verify JWTs locally instead of calling the provider per request; back off on 429 with jitter |
| Email / SMS sending | Supabase built-in email: 2 per hour; Firebase free plan: 150 password-reset emails per day; Cognito: limits on forgot-password requests per user per hour | your own transactional email provider; CAPTCHA before sending codes; prefer TOTP/passkeys over SMS |
| bcrypt input | 72 bytes | Argon2id instead, or pre-hash with HMAC + pepper as OWASP describes |
| Login attempts | NIST: at most 100 consecutive failures per account | progressive delays well before that; alert on spikes |
| JWKS fetches | providers rate-limit the JWKS endpoint too | cache by `kid`; refetch at most once per minute on an unknown `kid` |

### 10.4 Production checklist

In [29]:
CHECKLIST = {
    "Secrets":        ["signing keys and client secrets in a secrets manager, not in code or env files in git",
                       "JWT HMAC keys >= 32 random bytes, or asymmetric keys (RS256/ES256/EdDSA)",
                       "pepper and TOTP secrets stored apart from the user table"],
    "Rotation":       ["signing keys rotate on a schedule with a JWKS overlap window",
                       "API keys and client secrets rotatable with two active at once",
                       "refresh tokens rotate with reuse detection"],
    "Login":          ["Argon2id/scrypt hashes, NIST length rules, breached-password check",
                       "per-account progressive delay + per-IP + global rate limits",
                       "passkeys or TOTP offered; step-up auth for sensitive changes"],
    "Sessions":       ["__Host- HttpOnly Secure SameSite cookies; ID rotated on login",
                       "idle + absolute timeouts; server-side logout; 'log out everywhere'"],
    "Authorization":  ["deny by default; object-level checks on every endpoint; tenant check first",
                       "authorization tests (IDOR cases) in CI; RLS as a second wall"],
    "Browser":        ["CORS allow-list; CSRF token or Fetch Metadata; strict CSP; frame-ancestors 'none'"],
    "Audit logs":     ["log login success/failure, MFA changes, password resets, role changes, key creation (never secrets)",
                       "logs are append-only and kept for your compliance period"],
    "Account takeover detection": ["alert on new device/location, impossible travel, many failed logins, MFA reset followed by email change",
                       "notify users by email on sensitive changes, with a 'this was not me' link"],
    "Privacy / GDPR": ["account deletion removes or anonymises identity data, sessions, tokens, and provider-side records",
                       "data export on request; minimal claims in tokens; IdP data processing agreement in place"],
}

# Score a (made-up) project against the checklist
done = {"Secrets": 3, "Rotation": 1, "Login": 2, "Sessions": 2, "Authorization": 1, "Browser": 1,
        "Audit logs": 1, "Account takeover detection": 0, "Privacy / GDPR": 1}
total = sum(len(v) for v in CHECKLIST.values())
print(f"{'area':<28} done / items")
for area, items in CHECKLIST.items():
    d = done.get(area, 0)
    print(f"{area:<28} {d} / {len(items)}  {'' if d == len(items) else '<- todo: ' + items[d][:70]}")
print(f"\nreadiness: {sum(done.values())}/{total} items ({100 * sum(done.values()) / total:.0f}%) for this made-up project")

area                         done / items
Secrets                      3 / 3  
Rotation                     1 / 3  <- todo: API keys and client secrets rotatable with two active at once
Login                        2 / 3  <- todo: passkeys or TOTP offered; step-up auth for sensitive changes
Sessions                     2 / 2  
Authorization                1 / 2  <- todo: authorization tests (IDOR cases) in CI; RLS as a second wall
Browser                      1 / 1  
Audit logs                   1 / 2  <- todo: logs are append-only and kept for your compliance period
Account takeover detection   0 / 2  <- todo: alert on new device/location, impossible travel, many failed logins, M
Privacy / GDPR               1 / 2  <- todo: data export on request; minimal claims in tokens; IdP data processing 

readiness: 12/20 items (60%) for this made-up project


## 11. Service map, how to try it for real, certification topic map

### 11.1 Service map: the same need on each cloud

| Need | AWS | Azure | Google Cloud | Open source / independent |
|---|---|---|---|---|
| Customer sign-in (CIAM) | Amazon Cognito user pools | Microsoft Entra External ID | Identity Platform (Firebase Authentication) | Keycloak, Ory, authentik, Zitadel, Supabase Auth |
| Employee SSO / workforce IdP | IAM Identity Center (federates to an IdP) | Microsoft Entra ID | Cloud Identity / Google Workspace | Keycloak, authentik |
| Fine-grained app authorization | Amazon Verified Permissions (Cedar) | Entra app roles (coarse) | no direct managed equivalent | OpenFGA, SpiceDB, OPA, Cedar, Casbin |
| Protect APIs with tokens | API Gateway authorizers (JWT, Lambda), ALB OIDC auth | API Management `validate-jwt` policy | API Gateway / Apigee, Identity-Aware Proxy | Envoy / Kong / Ory Oathkeeper |
| Workload identity (no stored keys) | IAM roles, EKS Pod Identity / IRSA | Managed identities, workload identity federation | Service accounts, Workload Identity Federation | SPIFFE / SPIRE |
| Secrets and signing keys | Secrets Manager, KMS | Key Vault | Secret Manager, Cloud KMS | HashiCorp Vault / OpenBao |
| Bot and credential-stuffing protection | AWS WAF (bot control, account takeover prevention) | Azure WAF, Front Door | Cloud Armor, reCAPTCHA | CAPTCHA services, rate limiters |

### 11.2 How to try it for real (free and local options)

- **Keycloak in Docker:** `docker run -p 8080:8080 -e KC_BOOTSTRAP_ADMIN_USERNAME=admin -e KC_BOOTSTRAP_ADMIN_PASSWORD=admin quay.io/keycloak/keycloak start-dev` gives you a real OIDC and SAML server on localhost. Create a realm, a client with PKCE, and log in from a small FastAPI app using Authlib.
- **OAuth playgrounds:** the OAuth 2.0 Playground at oauth.com and Google's OAuth 2.0 Playground let you step through the code flow with a real server.
- **Free developer tiers:** Auth0, Clerk, Supabase, Firebase, Cognito, WorkOS and Entra External ID all have free tiers for small projects (limits change: check before relying on them).
- **Passkeys locally:** browsers treat `http://localhost` as a secure context, so you can try `py_webauthn` or `@simplewebauthn` examples on your laptop; Chrome DevTools has a WebAuthn panel with virtual authenticators.
- **TOTP:** scan the `otpauth://` URI from section 3.1 (turned into a QR code with any QR library) with an authenticator app and compare codes.
- **Policy engines:** run `opa eval` on a Rego file, use the Cedar playground, or start OpenFGA in Docker and load the document-sharing model from section 8.
- **jwt.io-style debuggers:** decode tokens locally; never paste production tokens into websites.

### 11.3 Certification topic map

This notebook supports the identity and access management parts of several certifications (named generally; check each current exam guide):

| Certification area | Sections here |
|---|---|
| Cloud security specialty exams (AWS, Azure, Google Cloud): identity and access management, federation, workload identity, secrets | 5, 6, 7.4, 10, 11.1 |
| Microsoft identity and access administrator: Entra ID, SSO, conditional access, external identities, app registrations | 3, 5, 6, 10 |
| Vendor-neutral security fundamentals (for example CompTIA Security+): authentication methods, MFA, federation, access control models | 1, 2, 3, 6, 8 |
| Broad security management (for example CISSP): identity and access management domain | 1, 3, 6, 8 |
| Secure web development (OWASP ASVS / Top 10 based training): authentication, session management, access control | 2, 4, 8, 9 |

## 12. Recap

| Topic | Remember |
|---|---|
| authN vs authZ | who you are vs what you may do; authZ on every request |
| Password storage | Argon2id (or scrypt/bcrypt/PBKDF2 per OWASP), salt, optional pepper, constant-time compare |
| Password rules | NIST 800-63B-4: 15+ chars (8+ with MFA), allow 64+, no composition rules, no forced rotation, blocklist |
| Login abuse | per-account delays + per-IP/global limits; same error for unknown user and bad password |
| Reset / magic links / OTP | random, hashed at rest, short-lived, single use; revoke sessions after reset |
| MFA | passkeys > TOTP > SMS; recovery codes hashed and single use; step-up for sensitive actions |
| TOTP | HMAC(key, time/30) + dynamic truncation; +-1 step window, no replay, rate limit |
| Passkeys | public-key challenge-response bound to the site's origin: phishing resistant |
| Sessions | random ID in a `__Host-` HttpOnly Secure SameSite cookie; rotate on login; idle + absolute timeouts |
| JWTs | allow-list `alg`; verify signature, `iss`, `aud`, `exp`; short TTL; small payload; no secrets inside |
| Refresh tokens | rotate on every use, detect reuse, revoke the family |
| SPAs | BFF pattern keeps tokens off the browser |
| OAuth / OIDC | auth code + PKCE (S256) + state + nonce; client credentials for machines; device flow for TVs/CLIs; implicit and password grants are out |
| ID vs access token | ID token is for the client; access token is for the API |
| JWKS | verify by `kid`; rotate keys with overlap; cache |
| Enterprise SSO | OIDC or SAML per tenant; SCIM for deprovisioning; issuer-to-tenant check |
| API keys | prefix, hash, scopes, expiry, per-key limits, rotation with overlap |
| Signed requests / webhooks | HMAC over canonical request or raw body + timestamp + nonce/event ID |
| Service-to-service | workload identity, mTLS, OIDC federation; no long-lived keys |
| Authorization models | RBAC (roles), ABAC (attributes), ReBAC (relationships, Zanzibar); deny by default |
| Browser | CORS protects reads only; CSRF needs SameSite + tokens/Fetch Metadata; CSP vs XSS; frame-ancestors vs clickjacking |
| Providers | buy or self-host a proven IdP; own your authorization logic |

## Interview questions

1. **What is the difference between authentication and authorization, and which HTTP status codes go with each?**
   Authentication proves identity (401 when missing or invalid). Authorization decides if that identity may do the action (403, or 404 to hide existence).
2. **Why not store passwords with SHA-256 plus a salt?**
   SHA-256 is designed to be fast, so a stolen database can be guessed at billions of tries per second on GPUs. Use a slow, memory-hard function such as Argon2id or scrypt.
3. **What does NIST SP 800-63B-4 say about password complexity and rotation?**
   No composition rules and no periodic forced changes; require length (15+ for single-factor), check against breached/common lists, and force a change only on evidence of compromise.
4. **Why are passkeys phishing resistant while TOTP is not?**
   The browser puts the real origin into the signed data and the authenticator only has a key for the real site, so a look-alike domain gets nothing. A TOTP code typed into a fake site can be relayed to the real one within its 30-second window.
5. **Session cookie or JWT for a web app?**
   For a browser app with one backend, a server-side session in an HttpOnly cookie is simpler and revocable instantly. JWTs suit short-lived access tokens between services or from an IdP. Many systems use both.
6. **Name three JWT verification mistakes.**
   Trusting the `alg` header (including `none`), not checking `aud`/`iss`/`exp`, and using a weak HS256 secret that can be cracked offline. Also: long lifetimes and putting secrets in the payload.
7. **How does refresh-token rotation detect theft?**
   Each refresh returns a new token and invalidates the old one. If an old token is used again, two parties have it, so the server revokes the whole token family and forces a new login.
8. **What do PKCE, `state` and `nonce` each protect against?**
   PKCE: a stolen authorization code cannot be redeemed without the verifier. `state`: CSRF on the redirect (an attacker injecting their own code). `nonce`: replay of an ID token.
9. **Why were the implicit and password grants deprecated?**
   Implicit puts tokens in the URL (leaks via history and referrers) without PKCE or refresh. The password grant hands the user's password to the client and cannot support MFA, passkeys or SSO.
10. **Does CORS protect my API?**
    No. It only controls whether browser JavaScript from another origin may read responses. Non-browser clients ignore it, and simple cross-site requests (form POSTs) are still sent, so you need authentication, authorization and CSRF defences.
11. **What is an IDOR and how do you prevent it?**
    Insecure direct object reference: changing an ID in the URL returns someone else's data because the server only checked login. Prevent with object-level authorization checks on every request, tenant scoping, deny by default, tests, and RLS as a backstop.
12. **RBAC vs ReBAC: when do you need ReBAC?**
    When access depends on relationships to specific objects (sharing, folders, teams, orgs) rather than global roles. Zanzibar-style systems such as OpenFGA and SpiceDB model this with relation tuples.

## Further reading

Standards and guidance (all verified as of November 2026):
- RFC 6749, The OAuth 2.0 Authorization Framework: https://www.rfc-editor.org/rfc/rfc6749
- RFC 7636, Proof Key for Code Exchange (PKCE): https://www.rfc-editor.org/rfc/rfc7636
- RFC 9700, Best Current Practice for OAuth 2.0 Security (January 2025): https://www.rfc-editor.org/rfc/rfc9700
- The OAuth 2.1 Authorization Framework (Internet-Draft): https://datatracker.ietf.org/doc/draft-ietf-oauth-v2-1/
- RFC 8628, OAuth 2.0 Device Authorization Grant: https://www.rfc-editor.org/rfc/rfc8628
- RFC 9449, OAuth 2.0 Demonstrating Proof of Possession (DPoP): https://www.rfc-editor.org/rfc/rfc9449
- RFC 7519, JSON Web Token: https://www.rfc-editor.org/rfc/rfc7519 and RFC 8725, JWT Best Current Practices: https://www.rfc-editor.org/rfc/rfc8725
- RFC 6238, TOTP: https://www.rfc-editor.org/rfc/rfc6238 and RFC 4226, HOTP: https://www.rfc-editor.org/rfc/rfc4226
- RFC 6265, HTTP State Management Mechanism (cookies): https://www.rfc-editor.org/rfc/rfc6265
- RFC 7644, SCIM Protocol: https://www.rfc-editor.org/rfc/rfc7644
- OpenID Connect Core 1.0: https://openid.net/specs/openid-connect-core-1_0.html
- W3C Web Authentication Level 3: https://www.w3.org/TR/webauthn-3/ and FIDO Alliance on passkeys: https://fidoalliance.org/passkeys/
- NIST SP 800-63B-4, Authentication and Authenticator Management: https://pages.nist.gov/800-63-4/sp800-63b.html
- OWASP Cheat Sheets: Authentication https://cheatsheetseries.owasp.org/cheatsheets/Authentication_Cheat_Sheet.html , Session Management https://cheatsheetseries.owasp.org/cheatsheets/Session_Management_Cheat_Sheet.html , Password Storage https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html , JSON Web Token https://cheatsheetseries.owasp.org/cheatsheets/JSON_Web_Token_Cheat_Sheet.html , CSRF Prevention https://cheatsheetseries.owasp.org/cheatsheets/Cross-Site_Request_Forgery_Prevention_Cheat_Sheet.html
- Zanzibar: Google's Consistent, Global Authorization System (USENIX ATC 2019): https://research.google/pubs/zanzibar-googles-consistent-global-authorization-system/
- AWS Signature Version 4: https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_sigv.html
- Standard Webhooks: https://www.standardwebhooks.com/
- MDN: Set-Cookie https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Set-Cookie and CORS https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/CORS

Providers and tools:
- Auth0 rate limit policy: https://auth0.com/docs/troubleshoot/customer-support/operational-policies/rate-limit-policy
- Clerk, how Clerk works: https://clerk.com/docs/guides/how-clerk-works/overview
- Supabase Auth rate limits: https://supabase.com/docs/guides/auth/rate-limits
- Firebase Authentication limits: https://firebase.google.com/docs/auth/limits
- Amazon Cognito quotas: https://docs.aws.amazon.com/cognito/latest/developerguide/quotas.html and refresh tokens: https://docs.aws.amazon.com/cognito/latest/developerguide/amazon-cognito-user-pools-using-the-refresh-token.html
- Azure AD B2C status and Entra External ID: https://learn.microsoft.com/en-us/azure/active-directory-b2c/overview
- Keycloak: https://www.keycloak.org/ ; Ory: https://www.ory.sh/docs/welcome ; authentik: https://goauthentik.io/ ; WorkOS SSO: https://workos.com/docs/sso
- Better Auth: https://www.better-auth.com/docs/introduction ; Auth.js joins Better Auth: https://www.better-auth.com/blog/authjs-joins-better-auth ; Lucia: https://lucia-auth.com/
- GitHub OAuth apps (web and device flow): https://docs.github.com/en/apps/oauth-apps/building-oauth-apps/authorizing-oauth-apps
- Google OpenID Connect: https://developers.google.com/identity/openid-connect/openid-connect
- OpenFGA: https://www.cncf.io/projects/openfga/ ; SpiceDB: https://authzed.com/spicedb ; OPA: https://www.cncf.io/projects/open-policy-agent-opa/ ; Cedar: https://www.cedarpolicy.com/ ; Casbin: https://casbin.org/ ; Permify: https://github.com/Permify/permify

**Next:** apply section 8's test suite to your own API, then revisit [27 / 02](../27_AI_and_Cybersecurity/02_securing_vibe_coded_applications.ipynb) to see these fixes inside a whole (deliberately broken) app.